# Nimbus PA — fontes defasadas e SST global

Este notebook compara dois híbridos de precipitação mensal com o mesmo calendário de informações.
Um utiliza atmosfera, índices oceânicos, SEAS5 e CFSv2; o outro acrescenta **oito componentes principais
da temperatura da superfície do mar (SST)** aos dois componentes LightGBM.

O objetivo é verificar se a SST ainda contribui quando os produtos mensais não são tratados como
disponíveis imediatamente ao fechar o mês. É uma etapa de pesquisa, com arquivos consolidados.

## Calendário fixado antes do treinamento

| Informação para prever o mês T | Mês usado | Exemplo: janeiro/2007 |
|---|---|---|
| Atmosfera ERA5 consolidada | T−4 | Setembro/2006 |
| Índices NOAA derivados de HadISST | T−3 | Outubro/2006 |
| SST ERSSTv5 | T−2 | Novembro/2006 |
| Inicialização SEAS5/CFSv2 | T−1 | Dezembro/2006 |
| Chuva mais recente permitida no treinamento | T−4 do primeiro alvo do bloco | Setembro/2006 |

A avaliação cobre **sete blocos de 24 meses entre 2007 e 2020**. Em cada bloco, o modelo fica fixo;
as entradas são atualizadas mês a mês, usando o calendário acima. Isso não significa emitir todas
as 24 previsões em uma única data. O treinamento termina em setembro anterior ao início do bloco,
com no máximo 360 meses e início comum em março/1982. A primeira SST é janeiro/1982.

As climatologias de chuva, atmosfera e índices usam janelas móveis de 360 meses, cada qual nas
datas reais de sua fonte. SEAS5, CFSv2 e PCA usam os meses associados aos exemplos disponíveis.
A feature climatológica da chuva e o resíduo de treino excluem a própria observação (leave-one-out).

## Modelo e comparação

- Dois LightGBMs: 300 árvores, 31 folhas, taxa 0,05, semente 42 e 5.000 pontos por mês.
- Regressão ridge local entre as duas previsões sazonais, penalização 0,1.
- Mistura fixa: 37,5% de cada árvore e 25% da regressão. Escalas dos resíduos: 0,9 e 0,875.
- A regressão e as colunas originais de treinamento são idênticas entre os dois tratamentos.
- PCA: oito componentes, SVD completa, anomalias mensais, peso sqrt(cos(latitude)), sem padronizar
  por desvio de cada pixel. Máscara oceânica, climatologias, centro e componentes são ajustados só no treino.
- Não há busca de hiperparâmetros, pesos, quantidade de PCs ou defasagens nesta execução.

Cada versão é treinada novamente. Avaliamos todos os 78.561 pontos em todos os 168 meses:
13.198.248 erros por modelo. A métrica principal é sqrt(SSE/n), em mm/dia. MAE, viés, RMSE com
peso de área, meses, anos, blocos e faixas de latitude ajudam a localizar ganhos e perdas.
Os modelos ajustam e avaliam precipitação ERA5, uma reanálise, não pluviômetros independentes.

## O que a política resolve e o que ainda falta

As defasagens são **hipóteses conservadoras, não comprovantes de publicação**. ERA5T mensal chega
cerca de cinco dias após o mês; ERA5 final substitui o preliminar aproximadamente dois meses depois.
Usar T−4 deixa margem para a consolidação. HadISST adiciona o mês anterior ao último no início
de cada novo mês; T−3 deixa margem adicional. ERSST T−2 é uma hipótese de disponibilidade a
conferir em cada emissão, sem garantia universal de data neste notebook.

SEAS5 via C3S tem agenda nominal de publicação no dia 6, 12 UTC. **A agenda C3S não comprova
a publicação do agregado CFSv2 no endpoint IRI usado aqui.** Seus membros têm inicializações
anteriores a T, porém falta um arquivo histórico das datas de chegada desse agregado.

Os dados finais podem ter revisões posteriores e os hindcasts foram produzidos retrospectivamente.
Portanto, a comparação mede sensibilidade às defasagens, e não uma operação histórica certificada.
Uma futura emissão deverá comprovar a chegada e a versão de cada arquivo antes do instante
de previsão; a função de verificação bloqueia datas ausentes ou posteriores. Este notebook
**não emite previsão de produção**. A autenticação dos registros exige um coletor auditável.

2007–2020 já foi usado para desenvolver modelos; não é reserva inédita. O teste anterior de 2025
permanece arquivado e **não é carregado nesta etapa**. Não há decisão baseada no placar Kaggle.

## Execução no Kaggle

Anexe os mesmos quatro Inputs: dados oficiais WorCAP, `worcap-seas5-dados`, `worcap-cfsv2-dados`
e o snapshot que contém `ersstv5_4graus_198201_202411.nc`. Não precisa de token nem novos downloads.
Use CPU e o ambiente da referência: NumPy 2.0.2, pandas 2.3.3, xarray 2025.12.0, LightGBM 4.6.0.

Execute todas as células em ordem. A primeira materializa código e protocolo, a segunda executa
testes temporais sintéticos, e a seguinte treina e avalia. Cada bloco grava modelos, PCA,
climatologias, coeficientes ridge, previsões e auditorias. Retomar a mesma pasta exige hashes
idênticos de código, dados, ambiente e arquivos de cada bloco já concluído.

Os resultados ficam em `/kaggle/working/nimbus_defasagens_4_3_2`. Salvar uma versão concluída
com outputs preserva modelos e previsões; o ZIP de relatórios contém tabelas, documentação e auditorias.
Os artefatos da competição e o notebook anterior não são alterados.

## Fontes e rastreabilidade

- [ERA5 — documentação ECMWF](https://confluence.ecmwf.int/spaces/CKB/pages/76414402/ERA5%2Bdata%2Bdocumentation).
- [HadISST — Met Office](https://www.metoffice.gov.uk/hadobs/hadisst/).
- [ERSST — NOAA NCEI](https://www.ncei.noaa.gov/products/extended-reconstructed-sst).
  Mantemos ERSSTv5 por comparabilidade, embora a página atual já apresente v6.
- [C3S — dados e calendário](https://confluence.ecmwf.int/spaces/CKB/pages/104239050/Summary%2Bof%2Bavailable%2Bdata).
- [CFSv2 — IRI Data Library](https://iridl.ldeo.columbia.edu/SOURCES/.Models/.NMME/.NCEP-CFSv2/.HINDCAST/.PENTAD_SAMPLES_FULL/.prec/).

Fontes documentais consultadas em 28/09/2026. Os manifests existentes conferem produto, unidade,
grade e membros sazonais. Os cinco meses NOAA julho–novembro/1976 são recuperados dos arquivos
brutos preservados; o período sobreposto é conferido exatamente contra o snapshot original.
Não se atualiza silenciosamente nenhuma série ou versão de dados.


In [ ]:
from pathlib import Path
import json, hashlib, runpy
RAIZ = Path('/kaggle/working/nimbus_defasagens_4_3_2')
CODIGO = RAIZ/'codigo'
CODIGO.mkdir(parents=True, exist_ok=True)
ARQUIVOS = {'biblioteca.py': 'from pathlib import Path\nfrom datetime import datetime, timezone\nfrom time import perf_counter\nimport base64, gzip, io, json, hashlib, gc, platform, importlib.metadata\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\nimport lightgbm as lgb\nfrom sklearn.decomposition import PCA\nfrom threadpoolctl import threadpool_limits\nfrom IPython.display import display, Markdown, Image, FileLink\nDATASET, SYSTEM, SCHEMA = \'seasonal-monthly-single-levels\', \'51\', \'worcap_seas5_v1\'\nPASTA_SEAS5_MANUAL = PASTA_CFSV2_MANUAL = None\n\n\nPONTOS_POR_MES = 5000\nANOS_TREINO = 30\nARVORES = 300\nSEMENTE = 42\nCORTE_FINAL = \'2022-12-01\'\nFIM_DESENVOLVIMENTO = \'2020-12-01\'\nPARAMETROS = {\'objective\': \'regression\', \'metric\': \'rmse\', \'learning_rate\': 0.05, \'num_leaves\': 31, \'min_data_in_leaf\': 150, \'lambda_l2\': 10.0, \'feature_fraction\': 0.9, \'bagging_fraction\': 0.8, \'bagging_freq\': 1, \'num_threads\': 4, \'seed\': SEMENTE, \'deterministic\': True, \'force_col_wise\': True, \'verbosity\': -1}\nBLOCOS = [{\'nome\': \'H1\', \'corte\': \'2006-12-01\', \'inicio\': \'2007-01-01\'}, {\'nome\': \'H2\', \'corte\': \'2008-12-01\', \'inicio\': \'2009-01-01\'}, {\'nome\': \'H3\', \'corte\': \'2010-12-01\', \'inicio\': \'2011-01-01\'}, {\'nome\': \'H4\', \'corte\': \'2012-12-01\', \'inicio\': \'2013-01-01\'}, {\'nome\': \'A\', \'corte\': \'2014-12-01\', \'inicio\': \'2015-01-01\'}, {\'nome\': \'B\', \'corte\': \'2016-12-01\', \'inicio\': \'2017-01-01\'}, {\'nome\': \'C\', \'corte\': \'2018-12-01\', \'inicio\': \'2019-01-01\'}]\nVARIAVEIS = [\'cloud_cover\', \'geopotential_850\', \'rel_hum_850\', \'shum_850\', \'surface_pressure\', \'t2\', \'temperature_850\', \'u_850\', \'v_850\']\nINDICES_NOMES = [\'nino34\', \'nino12\', \'tna\', \'tsa\']\nFONTES_INDICES = {\'fonte\': \'NOAA Physical Sciences Laboratory\', \'captura_utc\': \'2026-09-21T23:00:40.933121+00:00\', \'unidade\': \'degC\', \'tipo\': \'indices mensais de anomalia de temperatura da superficie do mar\', \'interpretacao_temporal\': \'Versoes retrospectivas, com mes de observacao estritamente T-1; uso de dados externos confirmado pelo participante. Nao e arquivo de vintages em tempo real.\', \'inicio\': \'1976-12-01\', \'fim\': \'2024-11-01\', \'meses\': 576, \'colunas\': [\'time_origem\', \'nino34\', \'nino12\', \'tna\', \'tsa\'], \'fontes\': [{\'indice\': \'nino34\', \'url\': \'https://psl.noaa.gov/data/timeseries/month/data/nino34.long.anom.data\', \'arquivo_bruto\': \'nino34.data\', \'sha256_bruto\': \'46e501b91f624ef85b525d73fbb427f9d04d30411ba6e5c7cff579b35fb4ff01\', \'inicio_declarado\': 1870, \'fim_declarado\': 2026, \'rodape\': \'-99.99\\n  NINA34\\n 5N-5S 170W-120W \\n HadISST \\n  Anomaly from 1981-2010\\n https://psl.noaa.gov/data/timeseries/month/\\n  units=degC\'}, {\'indice\': \'nino12\', \'url\': \'https://psl.noaa.gov/data/timeseries/month/data/nino12.long.anom.data\', \'arquivo_bruto\': \'nino12.data\', \'sha256_bruto\': \'b13ad5ad085cd13e70ec5d82662b509d522204e38a8eefd133f14be270e51928\', \'inicio_declarado\': 1870, \'fim_declarado\': 2026, \'rodape\': \'-99.99\\n  NINA12\\n 0N-10S 270W-280E \\n HadISST \\n  Anomaly from 1981-2010\\n https://psl.noaa.gov/data/timeseries/month/\\n  units=degC\'}, {\'indice\': \'tna\', \'url\': \'https://psl.noaa.gov/data/correlation/tna.data\', \'arquivo_bruto\': \'tna.data\', \'sha256_bruto\': \'387c4125f377e57bcb683a849aefcd76da7d7f3cdcb2ac68ab086ceadf300e8b\', \'inicio_declarado\': 1948, \'fim_declarado\': 2026, \'rodape\': \'-99.99\\n  TNAa\\n Tropical Northern Atlantic Index\\n SST Anom 5.5N-23.5N; 15W-57.5W\\n  Using HadISST1.1: Climo 1991-2020\\n  NOAA/PSL\\n  https://psl.noaa.gov/data/timeseries/month/\'}, {\'indice\': \'tsa\', \'url\': \'https://psl.noaa.gov/data/correlation/tsa.data\', \'arquivo_bruto\': \'tsa.data\', \'sha256_bruto\': \'700c0dffda0f19c1254ba482808b345eaa0538f5bf09fc3bc025f46a5fbe3a69\', \'inicio_declarado\': 1948, \'fim_declarado\': 2026, \'rodape\': \'-99.99\\n  TSAa\\n  NOAA/PSL\\n  https://psl.noaa.gov/data/timeseries/month/\'}], \'sha256_csv\': \'b6f444fbec48681c1e5b46a5c22aa197ad91c7fbbdaff82c3ed3a9c63f7c15a4\', \'transformacao_no_modelo\': \'Recentragem mensal em cada fold, usando somente as 360 origens de treino; sem padronizacao, medias moveis ou indices adicionais.\'}\nINDICES_CSV_GZIP_BASE64 = \'H4sIAAAAAAACCm1cy44tuY3cz7ecGuj9+JqBF4bRC7eBmf5/zElGBMkse1N90YpSSqL4CJKqv/7459//51//+8c//v7Pz59//PmvPuw/tX3++vNvn7/+72//Ve9eP7X9lPop/73G90ebn5/vP8vzc+wHsL+jABwDXBsywOwEcIZpv1xHIJqm6EDg//aDMfv3IGI8iB9fwYl/O2QSUrp9wQabAfslZOE7fX1/FMyBvVQCNhcyPz5Pw3oJOADg09PGlv2s+sblFLbPadMvW1dtQNTCA7v4PUMsWyeXUe1IK1eAM507nxik8kVgzH4bX8RxHBfLttUV++3asWoiJBdbXbctN0xHAMVSbPq9n3/a904hQFLpmz+/37AdX00hoeAg5ohl3EvIIgQSOzgJ+84+hGxBTlwe3MLeCTmEQBprxKkOrfYKgt/EinCAhEA0dor2m50nYzIkpBJS7cBGTOUInmt5jqP1tG5b63XRlOkAXMZxCdAMWNzxY52VAEoGJ4nLyL1NIgYRGxKW4vXCcdeW+zFlsw35DbuuK+VR6/0oTIXCaAmSyVeDpDDpmt9QFbtjhQt9joXjUpTz3ONGnX/WiXGpSW+P9cEFg6ZxgspjtJtbW2izVuCm64ZIqVLPxTrFJUHTVPwUhgCcondfPwRmwn4A3UVl6iMd8nHKASsrxa3caARMFyU/MGkFOS6btXWWWIDZtGecJgufhQ3AbepaglSDdxEQaOIiRKqBY6z49wYQEFcNDML00VRPQniauKu00XHzH0RzA/xTeXt7hdo/iEqJPL9ikHbdEONEK0US9qLH5d+dkK7LTdP8EdqMzoOQ5cIme/X7fTTHfO23PaKtMISTiBWW4EdWEj+nJnHD1cNO44xxw2pIB5/HTWyQ0SZE0oEVx2DrZkGAcOGUnoRzP9SFB6FzhV/lhT/mB4kIsxXCb3FLWsiGl6fFhs2sPBAJpybTiQVbvPBA3sKBf2rdHc8DkdLAh9bQ/SUEtWbN8OQG3pUAqs1tH+0Zerc1w05xzUreaWsrtF+3u5+FV9nayIUP/p5zk/WFw+EnIJX2aH2jIuztTusBVAL2p3MXAPgMDYCv+ek0IfuGQnWKpD33q32vfwQbpRLAGb73t/F2mDpUjXdsYj2b6BbCnbienaKoj7A74iaGI4vj08cHFR9fsRjhAVAQ6wHM5eZha4Idetafw/4xx5g/ceJOdapjT2a8h4LMgtMMXeM2qSAWUsnbzGQluxSkwhDjt9tyd/IgdLevuQHdfgFG6McpEbf1Yi6KCE0xe7oNjxbC3I/QDoYM1f3OGESMVxijIPjDEPRByHQ997XqxGCe9BnZrotwjT5L7vdBuOlKurHMSekz541QMPzEqkRILrADWIIhFudww3XS3TWd7YsIl8sJK2mxaO1ENCJwqDOpCbRgJsncOHc4qNIIcenW8E3TJD2JkGjuisgRUzV9R7K555kE7vHUMKIzhPOcfKWALyCFEAlnwwfOEIBFZw9E0lm41YMmMu3HpXMN0YLM+HdcbXaIQCE6IC4fmL7TCJfrmeFY4OJuuJ7CHbve0ABrO4+leBArxAOp4WRBBHAdV4gHC8GFOsUiKyIkHviLM3iCCmFWSGfU+MChzSJE0hk79rtHqM4K6cBgQ4GM/FathK6lriC8izECEccdnFzttgOpHGdsjJOgLzUsJ1BwfItH9qvnVTqFDJN+qYoE0N3fx/pAcy796AMQr69wF7jmk1E4AeSg3/XrfkEssEPbfct3l5V0CSdyOD6cCFdylCC5Z7uTv+Zlp4/jFEXnOb9ZYjDQyuFN/zy0C4TPk8MHw+PgE4zxfPv07vNxGL0quq9cGwRQH5tWH5OjYW5dx2/xqwnJyN7gt52+T2xf8aXdwuDu6zoJNH6BKCuYO8Or7TQMpCCYO311A0FS2JCI+7i8w9r9IWDKX077iUDRTpiARQDi9EvHLl7jnL2aQak0KyUU1il7NUZSedftpH2fl4jH3VfR8eZ64IS92SlVOmzzUnUSUYmA51jJmBZB5ECgAHeHO0MILsr+QFJaaW0Pn8XZn7XCOkY4CPMj0s7dOH0xm12IGPJ12XKPcFOJtx9zHzAxsGQwHjcMFMNVpbHSd/bLyg0x44SQ90CQNJZuMC74Ta79xERwQAsIdx3jdzTkCJn9nvJ+ogyEvPkEbgADvGe7Nzg8LVCPn+sQkVinq8tx73KDxSsbGLkEcww38fgV89t9L5pierrC+UzZnn+6weRLiRiXi9FH5DhwNTaJ5w/V7hZ3HDCEK7GxrjnoOpgQOX4yrQEg14ET3Yk+V04hHl+Ox0CYovDIncannAPZoJ1GjcSKckye07NQ6dYQCkOSuCNmH26NJFfxQ+v7NcXwDJOHc7IzRMycjPUEoQRfg51Mp5vLY8tbgyU2Z4lrvxZJgYzui0QG0rKst7o8UgIBpmVyF+7KbZoW7GBeAuSIx0c6PVb+hLyJLRJmZUT2/DZ35StpwfCcym3uyjHDiLDFEhm3uSs3T8cYD9ZFnyBPHM+Pne7FOQSQKLYnWjhdF6LyUgVjn89JbucsFf78BmOHzlB/riePbktMMeUd5npBPNOV8vMwckb8b0tJ4BkkEHRMu40k8EhFCYTym5Dm+VNPcpEi2Zl21xHoaOIuEHx3FTHbTdMUub/bo26yPYajOR4EjJQFwaG35rz49gixnignZ5MtyLuJvj+X7yQK1vWJ7RnSt0Fo+sTxaFiqijtqPvz20JDlZsuumUUBt0ciePkXLI9gCaXb3VzBpDM9O0VDb3dRoNSCK7OLYr07woWMsNrKKxLhfijZ/518SGLuJSU393LCdYeLo1g403L1qBOhbPCIiH2X8BHDBcI83g2TZNWDO37RD1CYtcNgDBcJ7iMYDOgD/MyI5HzyVVhp515eDIRRVRT17nibrZlS8JVnKrNlUTms94ig685gIOXj7KV80jjlerYbb7OyuJoz1CMX0KrC5DvDgWzkhulNmeK408UBgYU3avrCennK/mFeFs54RnI+ygegEpr/vKIoC9QbrQ4Rb46ebEXlHG6uaHVrZGUbD8rNFW6sKRrDG37G4yxMYsSDZMwQiaOv6bkiHKxF2jdT9OXBYPNg/a7fuS2rj9kPS6DdRNC7MQLbrPFGSDTT88LMCu2KEZubyPljRypV2ajh1CL2y3Qj5LePWBH4riSUEr7WjI/v9L7KhDMZ1sJJXCqwSsgy8P4XQt7hL3IfgBcuNlInnclLd264YjsVTLpbRhYhJhGapEWcD9uwChHJdMliIJY5jQhqiiUhLGUAQ30qx6erYqOlRxb4cpxU/bv2jpokzPA9HCdX/4ZYHfwtFVJukPU7niTxiQpF0wcu8txf9Ry0V9XZyN2ei/9+doCG1vAh2zPxw9LYTSVRHZDn4Z/ZVSxlDvueyMI/SfIJmqSc7j2eg/9aifb4UDOH+vbx2Grc1zDu/Akf/hwNCoiwMxxWJeQ8w7BuItk3VddRpSgKJywNdFNpfXuew3IWiEFSXf08H71Dnn5ofqmCEdkelUwejSfeUXVoXmCCopzI7+LCtCpY5f4jvSu11eWEOiZyTkXMpadCiCYZ1dk5i5eNCM/u2kYSIy1CyDqdbEuPLzWT853Yqm38ELGihvDzrjsLIYGclDc0RBfipCQzi/tMBxFwX1PcLQDczn1VQyJIOcrq3htSGSkfUc8nDj3EYraa1c+GNGgrpYRYYNmQemcGuhHyTt2P8HGOUNIEYTiCeRTbJxEjMiKy1mZhvqoPwPwPZRmRCEI84z7j0OFjvtEKIPvliNWOQ4MOiDuOVCBA0FkEuaGO7hAA/4aABnHPgcsx/sOxhT+fESDitmm5Uaxqv+f6RgVfSCq1I6zHLei0rYS4S09pFTF2QpJTl9bJmRAxws/5TlBpJmC+HOEarufYcC6115zvQvoFCEVa1aP70uJAUp0d5gfdTOW1zptcdkVCFVG8PhJOHSH+cD9VOhH15fZhL1ntJqK9GhzM1DGtYZcxldkZoQwnd9+QD4iWM0msMDIoAECpE89dtCSSKLDX4b0tYHW4p1Ffb9EYMdAdw/HlXucn9e1pm0HWZ8owmuNoApCDkN3ky6ddkoMgRZpodOMulYM3T44P0bdwBlEQM/qw10hF6ahFQUa0CcG2nGc8KPo9nodqvocg6IuE101PLwR073Fy9avgqgSMV+PGLR6N4t721EFnEW2f4QS1CleOG6SROT9NsjOttIYyuKXB8ZM60izPbxzBt3FzXhLhx0GIYuPigjC+FtKjqM3Pe6fWVXiBU1ocbqnTqzMPq68HO2/Lw1c1kmG85ZwofnWEdU/MvBYvpIH+YAWJmC+mJ7w9UN+YubXprtRnKsTKR7xuZKQvAeotDQulkjYBFMJSN6c3ZXKcUljbQwKqJ8Ylhd2l+eR4Gq8pl2miUL4ewzzGXRX9sjr5DM/I6K6PhND8IgcXZwXGhQhVCS4O22o3fDJOxPjISRxjWyyZagLl171WdcgEOb48zyoVYtJDG5ADT1RnvhHn1d8Ed1XDJCU2DsK/UhGuAxFtcpPlM0/FcyNRMB8RkF5SbSBacu8qXS6ynS8ikXGEgKDrasQCwufw5Jz2RkCPwF+tlyXuw3q57R9VKVc4ifW7RQ7anaKQFfmR641HiO40Q+RHFIBGaBdl8u5NrHTKHL7pQu8T/APDrg9PiRZmARECh7Ntv5Hs0trkG0zRUDQ4vrZoezeuNne42Mvx5nUYXrZ+wgUnzl2nJyzVjEjESDlHXiVvDARipmiyIg2cY6+dfIPNwXReBNg71AJByB0pGBDkpNC3MlztKTLaUaktdqn7iHZeTuKcA4yCvObGbdlBOuaJUis9+iFEhAF9L6uH48WWz7/RjhKWH+7uBO1A8FGX16YhvRO0A46qS5AIHFIB3dRiFrcoiMDOuwdLlnZ/5C9y2/vw3Pf8yF0kYm4JSROtrQX27ryMlYpWzX3ueUWzSsaF8uRW9xKMqfoFT43uKrr5x3yK9moi443oHiYmZl5KpGzq9vDmvuxU1Emmx2CJmCM1ecRkjNwDkR6HKCge0EgCZmqGr9uDmDE4vjxR7A1Q153bjSrg/chzb4+eotMdRcIlSRcNX4+VPUvY07hn0pc71fKRR7kRxF4F7JAj9yYzdXwD6BZ8frsWz6Hbfs2t2uGOxWEZuaNW/wuvx2GmpqiE40PCMgfHRwoKpiXzJ5pEMDzzW4l+1QbUNL5+XjVZ5fJ9+p3a9mgBN+IiAs6rQcRy7+pRBEB2aUY7ed+6XbWEWVqTeQhGXY1HEEYJxu96HNsIcJOkHK3U0RZZk0EqHjQgRmsEtMhxKHy2A9N4f3n74RFwPwRIAdipNv2GCjCj71TPGIrb75oINuZdHtfYNaupk93OOFKgRWs8r+R/Kw4oBNxXU6OHYrS5NbFrNAHv+opIag1RlH+z7KYstYYwSnonxvdLttVEr2+PRAz7uxchLborPSzQiyQgfpUyUukFmteSVbrOANmXopXMaP2P/A0ffwCxEk+/15OnUKBg2eZDTrw+wfVvmVJIaKwycPym3zf178NzUbW5adKTk09q8QOg5vcx1iY09LIEgHcDz0pEzqboqYDRghh5FFh7evBRcl1JAUrtKZYanohAxss/MlLgXsk03UjXRLNBUOxAhju7mlg2e9FGZLKF2K8p+npnh2rqZGcN2g2+b/XGqywlqRhRc6uRguIxB+IQUV/vK2akwoxf1NTIrmTI66nHFzJeQlEEQBrTiXg/w2FXzHTeXXNFfEdfLmKlqlletK/NV6c6EDM9ydwjPxPD+ErPyFQTcuubquH9o146JvUuASfpWOLWvoubSAhSM13+twbrRvpzRM6QJykNOc4TCjtPOC7avbUGVSee8Zl5RoSqzqFqMG9w2ZWeJUwCXkmo/AoIe5i5xOQdRnEE03t3LADwnuT0+2rEfQpcNfo4jafU6dW9rxNuSh+3MCPTy3tWPizHGWPXB1jdQ5Xt/DJDM6p7VqQ7YouQwfTq3tzAqJSh32Z9zpaHvJhHh3V5dW8+PPI0j7A2h/nble9xPeLmcPfKZlXzgIhEXd43BXbnqTHc/vW+/fYDkVrl8PK8ldp8I/5Kte4241aUDDg5eY8S3RSLqanQbZa9TXGEyu9H78GM2J9d1ADUVwGheEUftzdVuKe7e4swEIGm8nb30mM6v1TcVhtTajIHoIexq6ofJQ2N2rZZYouge4Ql+1fiaSq3CYO8c95JycepvFaNF+VwWGOJi2H/+/fzv728lUqf9waQ9e6J0AKiXtTctUEQnMFlcIZ7rtY8dtshgzNZ0PMbY1OcVCrSixTlAAVor2zS6Km5dROSE05VtDO5rvyefEWdVbSSkPl6qTpWvIQq+tDKHbCoKe7mBvH8biQ8oQCNgEiHe0spwo5KwE13Qp2vadyf+A/PJJtJ6RrX64wSz5Pz/P4447EFyGOo6v4dv7m91n3Wdq2JB+RipJ5G0DhdwklXjylTAobnmr0+uZXxr8GoVzzTLJ7xr8Go14pq33Lmd38lwxEqXbcbQam9ZU/jjePXC1w/YtbFDVdwakSB20vkMBvxdhyvbpqzpsP11dffRqjRGXYeCbToOl8jWk1MYpPj7eV0u/Tecs0tOs57JPunH2ALZj2uVA3BtYbfamAmEfN3ApzQ1Uh3bGlsK6/80g8dlxk2B5xXXJ5Yp47gvijfmU75OEPu/fghf4GP4SEEoVPXhNTRbHPLZewwj8ySPjajJXZdorbF6KYR4XMMf2hFQtOJeKeYmM6rXkFriWLjRuG9y1KM0RLFHjJyWedaotgl/vqIOeOpnaS/qOBcxjzi0Tfe7YJ1e7lta6/39fc22k0NcgYIj31oOXW9OEN9dwWg8lkV0LRMsbenVHASdi8SwcajMQsq0T43CHB6fbyMbtzSslItsWs0sCDixJ/TaESM1ytKxvVVRr619IKpEaHwuhKwXss8Hnp07WMnz6sXi56ZbKndHPVA+2W0LQjgWfH4CypMvgAQKY8bT4dOWkN0eizvTbIrgaub+DV6IpB48tpsS/R6H/8zL4iwLgHt9ZdTqqcCLEnQMrvuclnDi8et59f6DXRzNYUfLVrMx8Yzc7aSDf22HurPJ3S/H6bcLX5q3alDe152fwNgXhdf20nMYk48RNTxdX/Et57Jiz8TNC/QujcQzOez1j50blq5P89n5yGLTLtzODKvFN2xP2Vikw/PvG42QXNj93JYrQf2evLo6QQOdThtqCotWJ90pdyDOF9ljfgO0+LKlmizbFJl0bRxfOU/ZIS/wXKVem1Bm/nswFOz/v2TX12hAhdnl55+1+pmeXlPQRu/Exnmm5a3RrSR8hjR8rBoA/4ffzwna7xKAAA=\'\nINDICES_CSV_SHA256 = \'b6f444fbec48681c1e5b46a5c22aa197ad91c7fbbdaff82c3ed3a9c63f7c15a4\'\nSCHEMA_CFSV2 = \'worcap_cfsv2_pentad_media_auditada_v1\'\nPOLITICA_CFSV2 = \'ensemble_completo_ou_excecao_201908_m17_auditada_v2\'\nFONTE_CFSV2 = \'https://iridl.ldeo.columbia.edu/SOURCES/.Models/.NMME/.NCEP-CFSv2/.HINDCAST/.PENTAD_SAMPLES_FULL/.prec/\'\nFEATURE_SEAS5 = \'seas5_tp_prevista_media_T_emitida_Tmenos1\'\nFEATURE_SEAS5_ANOM = \'seas5_tp_anomalia_mensal_treino_fold\'\nFEATURE_CFSV2 = \'cfsv2_tp_prevista_media_T_emitida_Tmenos1\'\nFEATURE_CFSV2_ANOM = \'cfsv2_tp_anomalia_mensal_treino_fold\'\nNOMES_MODELOS = [\'arvores_seas5\', \'arvores_multissistema\']\nFASES_CFSV2 = dict(desenvolvimento=(\'1982-02-01\', \'2020-12-01\', 467), somente_ajuste_final=(\'2021-01-01\', \'2022-12-01\', 24), teste=(\'2023-01-01\', \'2024-12-01\', 24))\nMOS_ALPHA = 0.1\nMOS_FRACAO = 0.25\n\ndef exigir(condicao, mensagem):\n    if not bool(condicao):\n        raise ValueError(mensagem)\n\ndef sha256(arquivo):\n    digest = hashlib.sha256()\n    with Path(arquivo).open(\'rb\') as stream:\n        for bloco in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            digest.update(bloco)\n    return digest.hexdigest()\n\ndef salvar_json(nome, conteudo):\n    caminho = SAIDA / nome\n    caminho.write_text(json.dumps(conteudo, indent=2, ensure_ascii=False, default=str, allow_nan=False), encoding=\'utf-8\')\n    return caminho\n\ndef localizar_dados(pasta_manual=None):\n    obrigatorios = (\'treino_tp.nc\', \'treino_tp_alvo.nc\', \'teste_features.nc\', \'sample_submission.csv\')\n    if pasta_manual is not None:\n        candidatas = [Path(pasta_manual)]\n    elif Path(\'/kaggle/input\').is_dir():\n        candidatas = sorted({p.parent for p in Path(\'/kaggle/input\').rglob(obrigatorios[0])})\n    else:\n        candidatas = [Path.cwd() / \'data\' / \'raw\', Path.cwd().parent / \'data\' / \'raw\']\n    validas = sorted({p.resolve() for p in candidatas if all(((p / nome).is_file() for nome in obrigatorios))})\n    exigir(len(validas) == 1, f\'Não foi encontrada uma única pasta com os quatro arquivos oficiais. Anexe a competição em Input > Add Input ou defina PASTA_DADOS_MANUAL. Pastas completas encontradas: {validas}\')\n    return validas[0]\n\ndef conferir_campo(campo, referencia, datas):\n    exigir(set(campo.dims) == {\'time\', \'lat\', \'lon\'}, \'Dimensões atmosféricas inesperadas.\')\n    for eixo in (\'lat\', \'lon\'):\n        exigir(np.array_equal(campo[eixo].values, referencia[eixo].values), f\'Grade atmosférica desalinhada: {eixo}.\')\n    exigir(pd.DatetimeIndex(campo.time.values).equals(pd.DatetimeIndex(datas)), \'Calendário atmosférico desalinhado.\')\n\ndef ajustar_clima_atmosfera(campos, origens):\n    origens = pd.DatetimeIndex(origens)\n    resultado = {}\n    for nome in VARIAVEIS:\n        campo = campos[nome]\n        indices = pd.DatetimeIndex(campo.time.values).get_indexer(origens)\n        exigir((indices >= 0).all(), \'Meses de ajuste atmosférico ausentes.\')\n        mapas = []\n        for mes in range(1, 13):\n            ids = indices[origens.month == mes]\n            exigir(len(ids) > 0, \'Mês sem histórico atmosférico.\')\n            valores = campo.values[ids]\n            exigir(np.isfinite(valores).all(), f\'{nome}: valores de treino não finitos.\')\n            mapas.append(valores.mean(axis=0, dtype=np.float64).astype(np.float32))\n        resultado[nome] = np.stack(mapas)\n    return resultado\n\ndef reconstruir_chuva(clima_tp, anomalias, peso):\n    peso = float(peso)\n    exigir(np.isfinite(peso) and peso >= 0, \'Peso deve ser finito e não negativo.\')\n    meses = pd.DatetimeIndex(anomalias.time.values).month.to_numpy() - 1\n    valores = np.maximum(0, clima_tp.values[meses] + peso * anomalias.values)\n    exigir(np.isfinite(valores).all(), \'Reconstrução da chuva produziu valores inválidos.\')\n    return xr.DataArray(valores.astype(np.float32), dims=anomalias.dims, coords=anomalias.coords, name=\'tp_mm_day\', attrs={\'units\': \'mm/day\'})\n\ndef auditar_chuva_desenv():\n    with xr.open_dataset(PASTA / \'treino_tp.nc\') as ds, xr.open_dataset(PASTA / \'treino_tp_alvo.nc\') as da:\n        bruto, alvo = (ds.tp, da.tp_alvo)\n        esperado = pd.date_range(\'1940-01-01\', CORTE_FINAL, freq=\'MS\')\n        exigir(set(bruto.dims) == set(alvo.dims) == {\'time\', \'lat\', \'lon\'}, \'Dimensões inesperadas da precipitação.\')\n        bruto, alvo = (bruto.transpose(\'time\', \'lat\', \'lon\'), alvo.transpose(\'time\', \'lat\', \'lon\'))\n        exigir(bruto.shape == alvo.shape == (996, 301, 261), \'Grade/calendário inesperados.\')\n        exigir(pd.DatetimeIndex(bruto.time.values).equals(esperado), \'Calendário oficial divergente.\')\n        exigir(bruto.attrs.get(\'units\') == alvo.attrs.get(\'units\') == \'mm/day\', \'Chuva deve estar em mm/day; nenhuma conversão será aplicada.\')\n        for eixo in [\'time\', \'lat\', \'lon\']:\n            exigir(np.array_equal(bruto[eixo].values, alvo[eixo].values), f\'Alvo desalinhado em {eixo}.\')\n        for eixo in [\'lat\', \'lon\']:\n            exigir(np.all(np.diff(bruto[eixo].values) == 0.25), f\'Grade inválida: {eixo}.\')\n        chuva = bruto.sel(time=slice(None, FIM_DESENVOLVIMENTO)).load()\n        exigir(np.isfinite(chuva.values).all() and (chuva.values >= 0).all(), \'Precipitação histórica inválida.\')\n        for i in range(0, chuva.sizes[\'time\'] - 1, 24):\n            j = min(i + 24, chuva.sizes[\'time\'] - 1)\n            exigir(np.array_equal(alvo.isel(time=slice(i, j)).values, chuva.values[i + 1:j + 1]), \'tp_alvo[M] não coincide com tp[M+1].\')\n    return chuva\n\ndef carregar_atmosfera(referencia, inicio, fim):\n    resultado, metadados = ({}, [])\n    esperado_completo = pd.date_range(\'1940-01-01\', CORTE_FINAL, freq=\'MS\')\n    esperado_recorte = pd.date_range(inicio, fim, freq=\'MS\')\n    for nome in VARIAVEIS:\n        with xr.open_dataset(PASTA / f\'treino_{nome}.nc\') as ds:\n            bruto = ds[nome].transpose(\'time\', \'lat\', \'lon\')\n            conferir_campo(bruto, referencia, esperado_completo)\n            campo = bruto.sel(time=slice(inicio, fim)).astype(\'float32\').load()\n            conferir_campo(campo, referencia, esperado_recorte)\n            exigir(np.isfinite(campo.values).all(), f\'{nome}: atmosfera não finita.\')\n            resultado[nome] = campo\n            metadados.append(dict(variavel=nome, atributos=dict(bruto.attrs), inicio=str(inicio), fim=str(fim), minimo=float(campo.min()), maximo=float(campo.max())))\n        print(\'Carregado:\', nome, campo.shape, flush=True)\n    return (resultado, metadados)\n\ndef carregar_indices_incorporados():\n    dados = gzip.decompress(base64.b64decode(INDICES_CSV_GZIP_BASE64))\n    exigir(hashlib.sha256(dados).hexdigest() == INDICES_CSV_SHA256, \'Hash das series NOAA incorporadas divergente.\')\n    tabela = pd.read_csv(io.BytesIO(dados), parse_dates=[\'time_origem\'])\n    exigir(tabela.columns.tolist() == [\'time_origem\'] + INDICES_NOMES, \'Colunas dos indices oceanicos divergentes.\')\n    tabela = tabela.set_index(\'time_origem\')\n    conferir_indices(tabela)\n    esperado = pd.date_range(FONTES_INDICES[\'inicio\'], FONTES_INDICES[\'fim\'], freq=\'MS\')\n    exigir(tabela.index.equals(esperado), \'Snapshot NOAA tem lacunas ou datas alteradas.\')\n    return (tabela, dados)\n\ndef conferir_indices(tabela):\n    exigir(tabela.columns.tolist() == INDICES_NOMES, \'Ordem dos quatro indices alterada.\')\n    datas = pd.DatetimeIndex(tabela.index)\n    exigir(len(datas) > 0 and datas.is_unique and datas.is_monotonic_increasing, \'Datas NOAA vazias, duplicadas ou fora de ordem.\')\n    exigir(datas.equals(pd.date_range(datas.min(), datas.max(), freq=\'MS\')), \'Calendario NOAA precisa ser mensal, continuo e no primeiro dia.\')\n    exigir(np.isfinite(tabela.to_numpy(dtype=np.float64)).all(), \'Indice NOAA ausente/infinito. Nao interpolar nem preencher com meses futuros.\')\n\ndef auditar_dataset(ds, esperado=None):\n    exigir(ds.attrs.get(\'schema\') == SCHEMA, \'Versão de artefato SEAS5 desconhecida.\')\n    exigir(ds.attrs.get(\'dataset\') == DATASET and str(ds.attrs.get(\'system\')) == SYSTEM, \'Proveniência SEAS5 inválida.\')\n    exigir(ds.attrs.get(\'product_type\') == \'monthly_mean\' and int(ds.attrs.get(\'leadtime_month\')) == 2, \'Produto ou lead incorreto.\')\n    exigir(ds.seas5_tp_media.attrs.get(\'units\') == \'mm/day\', \'SEAS5 deve estar em mm/day após conversão.\')\n    exigir(ds.seas5_tp_media.dims == (\'time\', \'lat\', \'lon\'), \'Dimensões SEAS5 inesperadas.\')\n    datas = pd.DatetimeIndex(ds.time.values)\n    exigir(datas.is_unique and datas.is_monotonic_increasing, \'Alvos duplicados/desordenados.\')\n    exigir(datas.equals(pd.date_range(datas.min(), datas.max(), freq=\'MS\')), \'Alvos SEAS5 com lacunas.\')\n    if esperado is not None:\n        exigir(datas.equals(pd.DatetimeIndex(esperado)), \'Calendário SEAS5 não coincide com a requisição.\')\n    exigir(pd.DatetimeIndex(ds.time_origem.values).equals((datas.to_period(\'M\') - 1).to_timestamp()), \'SEAS5 não está inicializado exatamente em T−1.\')\n    exigir(np.isin(ds.n_membros.values, [25, 51]).all(), \'Conjunto de membros incompleto.\')\n    for eixo in [\'lat\', \'lon\']:\n        v = ds[eixo].values\n        exigir(len(v) > 1 and np.all(np.diff(v) == 1), f\'Grade nativa de 1 grau incorreta: {eixo}.\')\n    exigir(ds.lat.min() <= -60 and ds.lat.max() >= 15 and (ds.lon.min() <= -90) and (ds.lon.max() >= -25), \'Recorte não cobre a grade oficial. Não extrapolar.\')\n    exigir(np.isfinite(ds.seas5_tp_media.values).all() and (ds.seas5_tp_media.values >= 0).all(), \'Valores SEAS5 inválidos.\')\n\ndef localizar_seas5():\n    if PASTA_SEAS5_MANUAL:\n        candidatos = [Path(PASTA_SEAS5_MANUAL) / \'seas5_51_manifesto.json\']\n    else:\n        candidatos = []\n        for raiz in [Path(\'/kaggle/input\'), Path(\'/kaggle/working/worcap_SEAS5_dados\'), Path.cwd() / \'outputs/worcap_SEAS5_dados\']:\n            if raiz.is_dir():\n                candidatos.extend(raiz.rglob(\'seas5_51_manifesto.json\'))\n    candidatos = sorted(set((p.resolve() for p in candidatos if p.is_file())))\n    exigir(len(candidatos) == 1, \'Anexe o dataset preparado SEAS5 em Input, ou defina PASTA_SEAS5_MANUAL. É necessário encontrar exatamente um manifesto SEAS5.\')\n    caminho = candidatos[0]\n    m = json.loads(caminho.read_text(encoding=\'utf-8\'))\n    exigir(m.get(\'schema\') == SCHEMA and m.get(\'system\') == SYSTEM and (m.get(\'dataset\') == DATASET), \'Manifesto SEAS5 incompatível.\')\n    exigir(m.get(\'leadtime_month\') == 2 and m.get(\'paramId\') == 172228 and (m.get(\'product_type\') == \'monthly_mean\') and (m.get(\'variable\') == \'total_precipitation\'), \'Produto SEAS5 não corresponde ao experimento.\')\n    return (caminho.parent, m)\n\ndef carregar_seas5(uso, referencia):\n    item = MANIFESTO_SEAS5[\'arquivos\'][uso]\n    caminho = PASTA_SEAS5 / item[\'arquivo\']\n    exigir(caminho.is_file() and sha256(caminho) == item[\'sha256\'], f\'SEAS5 ausente/alterado: {uso}.\')\n    with xr.open_dataset(caminho) as ds:\n        bruto = ds.load()\n    auditar_dataset(bruto, pd.date_range(item[\'inicio\'], item[\'fim\'], freq=\'MS\'))\n    exigir(bruto.attrs.get(\'uso\') == uso, \'Uso do arquivo SEAS5 divergente.\')\n    grade = bruto.seas5_tp_media.interp(lat=referencia.lat, lon=referencia.lon, method=\'linear\').astype(np.float32)\n    exigir(np.isfinite(grade.values).all() and (grade.values >= 0).all(), \'Interpolação SEAS5 inválida.\')\n    for coord in [\'lat\', \'lon\']:\n        exigir(np.array_equal(grade[coord].values, referencia[coord].values), \'Grade SEAS5 desalinhada.\')\n    grade.attrs.update(units=\'mm/day\', system=SYSTEM, leadtime_month=2)\n    return grade\n\ndef valores_seas5(seas, destino, pontos=None):\n    destino = pd.Timestamp(destino)\n    exigir(pd.DatetimeIndex(seas.time.values).is_unique, \'Alvos SEAS5 duplicados.\')\n    i = pd.DatetimeIndex(seas.time.values).get_indexer([destino])[0]\n    exigir(i >= 0, f\'Previsão SEAS5 ausente para {destino.date()}.\')\n    origem = pd.Timestamp(seas.time_origem.values[i])\n    exigir(origem == (destino.to_period(\'M\') - 1).to_timestamp() and origem < destino, \'SEAS5 usa inicialização incorreta ou posterior ao limite.\')\n    v = seas.values[i].reshape(-1)\n    if pontos is not None:\n        v = v[pontos]\n    exigir(np.isfinite(v).all(), \'Feature SEAS5 não finita.\')\n    return v\n\ndef amostrar_m5_documentado(chuva, atmosfera, seas, corte):\n    destinos = calendario_pareado(corte)\n    origens = (destinos.to_period(\'M\') - 1).to_timestamp()\n    ref_origens, ref_alvos = calendario_pares(atmosfera[VARIAVEIS[0]].time.values, corte, 30)\n    clima, somas, contagens = estatisticas_climatologia(chuva, corte, 30)\n    exigir(np.all(contagens == 30), \'Climatologia de chuva não tem exatamente 30 anos.\')\n    ca = ajustar_clima_atmosfera(atmosfera, ref_origens)\n    ca[\'_clima_indices\'] = ajustar_clima_indices(INDICES_OC, ref_origens, corte)\n    exigir(set(destinos).issubset(set(ref_alvos)), \'Exemplo fora da janela de referência.\')\n    ngrade = chuva.sizes[\'lat\'] * chuva.sizes[\'lon\']\n    n = min(PONTOS_POR_MES, ngrade)\n    X = np.empty((len(destinos) * n, 28), dtype=np.float32)\n    y = np.empty(len(X), dtype=np.float32)\n    ids = np.empty((len(destinos), n), dtype=np.int32)\n    for i, t in enumerate(destinos):\n        pontos = np.random.default_rng(np.random.SeedSequence([SEMENTE, t.year, t.month])).choice(ngrade, size=n, replace=False)\n        ids[i] = pontos\n        basicas = matriz_mes(atmosfera, ca, clima, t, pontos)\n        obs = chuva.sel(time=t).values.reshape(-1)[pontos]\n        basicas[:, 22] = ((somas[t.month - 1].reshape(-1)[pontos] - obs.astype(np.float64)) / 29).astype(np.float32)\n        sl = slice(i * n, (i + 1) * n)\n        X[sl, :27] = basicas\n        X[sl, 27] = valores_seas5(seas, t, pontos)\n        y[sl] = obs - basicas[:, 22]\n    exigir(np.isfinite(X).all() and np.isfinite(y).all(), \'Treino não finito.\')\n    exigir((origens < destinos).all() and destinos.max() <= pd.Timestamp(corte), \'Corte temporal violado.\')\n    return (X, y, ids, destinos, clima, ca, somas, contagens)\n\ndef valores_anomalia_seas5(seas, clima_seas, destino, pontos=None):\n    for eixo in [\'lat\', \'lon\']:\n        exigir(np.array_equal(seas[eixo].values, clima_seas[eixo].values), \'Climatologia SEAS5 desalinhada.\')\n    bruto = valores_seas5(seas, destino, pontos)\n    referencia = clima_seas.sel(month=pd.Timestamp(destino).month).values.reshape(-1)\n    if pontos is not None:\n        referencia = referencia[pontos]\n    anomalia = (bruto - referencia).astype(np.float32)\n    exigir(np.isfinite(anomalia).all(), \'Anomalia SEAS5 não finita.\')\n    return anomalia\n\ndef amostrar_arvores_seas5(chuva, atmosfera, seas, corte):\n    x5, y, ids, destinos, clima, ca, somas, contagens = amostrar_m5_documentado(chuva, atmosfera, seas, corte)\n    clima_seas = ajustar_clima_seas5(seas, corte)\n    X = np.empty((len(x5), 29), dtype=np.float32)\n    X[:, :28] = x5\n    n = ids.shape[1]\n    for i, t in enumerate(destinos):\n        X[i * n:(i + 1) * n, 28] = valores_anomalia_seas5(seas, clima_seas, t, ids[i])\n    exigir(np.array_equal(X[:, :28], x5), \'As 28 features do base_seas5 foram modificadas.\')\n    exigir(np.isfinite(X).all() and np.isfinite(y).all(), \'Treino não finito.\')\n    del x5\n    return (X, y, ids, destinos, clima, ca, somas, contagens, clima_seas)\n\ndef validar_manifesto_cfsv2(m):\n    exigir(m.get(\'schema\') == SCHEMA_CFSV2 and m.get(\'modelo\') == \'NCEP-CFSv2\' and (m.get(\'fonte\') == FONTE_CFSV2), \'Manifesto CFSv2 incompatível.\')\n    exigir(m.get(\'lead_iri\') == 1.5 and m.get(\'unidade\') == \'mm/day\' and (m.get(\'chuva_observada_utilizada\') is False), \'Produto ou temporalidade CFSv2 incorretos.\')\n    exigir(m.get(\'meses\') == 515 and m.get(\'inicio_alvos\') == \'1982-02-01\' and (m.get(\'fim_alvos\') == \'2024-12-01\'), \'Cobertura CFSv2 divergente.\')\n    exigir(m.get(\'politica_membros\') == POLITICA_CFSV2 and m.get(\'excecoes_permitidas\') == {\'2019-08-01\': [17]}, \'Política de membros CFSv2 divergente.\')\n    aplicadas = m.get(\'excecoes_aplicadas\', [])\n    exigir(isinstance(aplicadas, list) and len(aplicadas) <= 1, \'Exceções CFSv2 não previstas.\')\n    if aplicadas:\n        r = aplicadas[0]\n        exigir(r.get(\'time_origem\') == \'2019-08-01\' and r.get(\'time_alvo\') == \'2019-09-01\' and (r.get(\'membros\') == 23) and (r.get(\'membros_esperados\') == 24) and (r.get(\'membros_indisponiveis_ids\') == [17]), \'Exceção CFSv2 incorreta.\')\n        a = r.get(\'auditoria_excecao\', {})\n        exigir(a.get(\'membros_indisponiveis_ids\') == [17] and np.isfinite(a.get(\'maximo_delta_media_mm_day\', np.nan)) and (0 <= a[\'maximo_delta_media_mm_day\'] <= 1e-05), \'Exceção sem média individual conferida.\')\n    for fase, (inicio, fim, n) in FASES_CFSV2.items():\n        item = m.get(\'arquivos\', {}).get(fase, {})\n        exigir(item.get(\'arquivo\') == f\'cfsv2_{fase}.nc\' and item.get(\'inicio_alvos\') == inicio and (item.get(\'fim_alvos\') == fim) and (item.get(\'meses\') == n), \'Partição CFSv2 incorreta.\')\n\ndef localizar_cfsv2():\n    if PASTA_CFSV2_MANUAL:\n        candidatos = [Path(PASTA_CFSV2_MANUAL) / \'cfsv2_manifesto.json\']\n    else:\n        candidatos = []\n        for raiz in [Path(\'/kaggle/input\'), Path(\'/kaggle/working/worcap_CFSv2_dados\'), Path.cwd() / \'outputs/worcap_CFSv2_dados\']:\n            if raiz.is_dir():\n                candidatos.extend(raiz.rglob(\'cfsv2_manifesto.json\'))\n    candidatos = sorted(set((p.resolve() for p in candidatos if p.is_file())))\n    exigir(len(candidatos) == 1, \'Anexe a saída concluída do notebook CFSv2 em Input > Notebook ou defina PASTA_CFSV2_MANUAL; necessário exatamente um manifesto CFSv2.\')\n    caminho = candidatos[0]\n    m = json.loads(caminho.read_text(encoding=\'utf-8\'))\n    validar_manifesto_cfsv2(m)\n    return (caminho.parent, m)\n\ndef auditar_cfsv2(ds, uso, m):\n    inicio, fim, n = FASES_CFSV2[uso]\n    datas = pd.date_range(inicio, fim, freq=\'MS\')\n    origens = (datas.to_period(\'M\') - 1).to_timestamp()\n    exigir(ds.attrs.get(\'schema\') == SCHEMA_CFSV2 and ds.attrs.get(\'modelo\') == \'NCEP-CFSv2\' and (ds.attrs.get(\'fonte\') == FONTE_CFSV2) and (ds.attrs.get(\'lead_iri\') == 1.5), \'Proveniência do NetCDF CFSv2 divergente.\')\n    exigir(ds.attrs.get(\'politica_membros\') == POLITICA_CFSV2, \'NetCDF CFSv2 sem política corrigida.\')\n    exigir(ds.cfsv2_tp_media.dims == (\'time\', \'lat\', \'lon\') and ds.cfsv2_tp_media.attrs.get(\'units\') == \'mm/day\', \'Dimensões/unidade CFSv2 inválidas.\')\n    exigir(pd.DatetimeIndex(ds.time.values).equals(datas), \'Datas CFSv2 faltantes, extras ou desordenadas.\')\n    exigir(pd.DatetimeIndex(ds.time_origem.values).equals(origens), \'CFSv2 não foi emitido em T-1.\')\n    exigir(np.array_equal(ds.lat.values, np.arange(-61, 17)) and np.array_equal(ds.lon.values, np.arange(-91, -23)), \'Grade nativa CFSv2 divergente.\')\n    for var in [\'n_membros\', \'inicializacao_mais_antiga\', \'inicializacao_mais_recente\', \'fase_fonte\', \'time_origem\']:\n        exigir(ds[var].dims == (\'time\',), f\'Coordenada CFSv2 inválida: {var}.\')\n    dmin = pd.DatetimeIndex(ds.inicializacao_mais_antiga.values)\n    dmax = pd.DatetimeIndex(ds.inicializacao_mais_recente.values)\n    exigir(not dmin.hasnans and (not dmax.hasnans) and (dmin <= dmax).all() and (dmax < datas).all() and (dmin >= origens - pd.Timedelta(days=40)).all() and (dmax <= origens + pd.Timedelta(days=7)).all(), \'Inicializações CFSv2 fora da janela permitida.\')\n    esperados = np.where(origens.month == 11, 28, 24)\n    if m[\'excecoes_aplicadas\'] and pd.Timestamp(\'2019-09-01\') in datas:\n        esperados[datas == pd.Timestamp(\'2019-09-01\')] = 23\n    exigir(np.array_equal(ds.n_membros.values, esperados), \'Número de membros CFSv2 difere do manifesto.\')\n    exigir(np.array_equal(ds.fase_fonte.values, (origens >= pd.Timestamp(\'2011-04-01\')).astype(np.int8)), \'Transição hindcast/operacional CFSv2 incorreta.\')\n    v = ds.cfsv2_tp_media.values\n    exigir(np.isfinite(v).all() and (v >= 0).all(), \'Precipitação prevista CFSv2 inválida.\')\n\ndef carregar_cfsv2(uso, referencia):\n    item = MANIFESTO_CFSV2[\'arquivos\'][uso]\n    path = PASTA_CFSV2 / item[\'arquivo\']\n    exigir(path.is_file() and sha256(path) == item[\'sha256\'], f\'CFSv2 ausente/alterado: {uso}.\')\n    with xr.open_dataset(path) as d:\n        bruto = d.load()\n    auditar_cfsv2(bruto, uso, MANIFESTO_CFSV2)\n    grade = bruto.cfsv2_tp_media.interp(lat=referencia.lat, lon=referencia.lon, method=\'linear\').astype(np.float32)\n    exigir(np.isfinite(grade.values).all() and (grade.values >= 0).all(), \'Interpolação CFSv2 inválida; não extrapolar.\')\n    for eixo in [\'lat\', \'lon\']:\n        exigir(np.array_equal(grade[eixo].values, referencia[eixo].values), \'Grade CFSv2 desalinhada.\')\n    for coord in [\'time_origem\', \'n_membros\', \'inicializacao_mais_antiga\', \'inicializacao_mais_recente\', \'fase_fonte\']:\n        grade = grade.assign_coords({coord: (\'time\', bruto[coord].values)})\n    grade.attrs.update(units=\'mm/day\', lead_iri=1.5)\n    salvar_json(f\'auditoria_CFSv2_{uso}.json\', dict(arquivo=str(path), sha256=item[\'sha256\'], meses=bruto.sizes[\'time\'], minimo=float(grade.min()), maximo=float(grade.max()), membros_usados=np.unique(bruto.n_membros.values).tolist(), interpolacao=\'bilinear sem extrapolacao\'))\n    return grade\n\ndef ajustar_clima_previsao(previsoes, corte, nome):\n    treino = calendario_pareado(corte)\n    datas = pd.DatetimeIndex(previsoes.time.values)\n    exigir(datas.is_unique and datas.is_monotonic_increasing and treino.isin(datas).all(), f\'Calendário incompleto/desordenado: {nome}.\')\n    sel = previsoes.sel(time=treino)\n    exigir(pd.DatetimeIndex(sel.time_origem.values).equals((treino.to_period(\'M\') - 1).to_timestamp()), f\'Origem incorreta na climatologia {nome}.\')\n    v = sel.values\n    exigir(np.isfinite(v).all() and (v >= 0).all(), f\'Valores inválidos na climatologia {nome}.\')\n    n = np.array([(treino.month == m).sum() for m in range(1, 13)], dtype=np.int16)\n    exigir((n > 0).all() and n.max() <= 30, \'Climatologia de previsão excede 30 anos ou mês ausente.\')\n    medias = np.stack([v[treino.month == m].mean(axis=0, dtype=np.float64) for m in range(1, 13)]).astype(np.float32)\n    return xr.DataArray(medias, dims=(\'month\', \'lat\', \'lon\'), coords={\'month\': np.arange(1, 13), \'lat\': previsoes.lat, \'lon\': previsoes.lon, \'n_meses\': (\'month\', n)}, name=f\'climatologia_{nome}\', attrs=dict(units=\'mm/day\', inicio=str(treino[0].date()), corte=str(pd.Timestamp(corte).date()), meses_treino=len(treino), anos_min=int(n.min()), anos_max=int(n.max()), referencia=\'somente preditores nos meses do treino pareado; uma media de ensemble por ano/mes\'))\n\ndef ajustar_clima_seas5(seas, corte):\n    return ajustar_clima_previsao(seas, corte, \'SEAS5\')\n\ndef valores_cfsv2(cfs, destino, pontos=None):\n    destino = pd.Timestamp(destino)\n    exigir(pd.Timestamp(cfs.inicializacao_mais_recente.sel(time=destino).values) < destino, \'Inicialização real CFSv2 invade o alvo.\')\n    return valores_seas5(cfs, destino, pontos)\n\ndef valores_anomalia_cfsv2(cfs, clima_cfs, destino, pontos=None):\n    for eixo in [\'lat\', \'lon\']:\n        exigir(np.array_equal(cfs[eixo], clima_cfs[eixo]), \'Climatologia CFSv2 desalinhada.\')\n    bruto = valores_cfsv2(cfs, destino, pontos)\n    ref = clima_cfs.sel(month=pd.Timestamp(destino).month).values.ravel()\n    if pontos is not None:\n        ref = ref[pontos]\n    anom = (bruto - ref).astype(np.float32)\n    exigir(np.isfinite(anom).all(), \'Anomalia CFSv2 inválida.\')\n    return anom\n\ndef amostrar_arvores_multissistema(chuva, atmosfera, seas, cfs, corte):\n    x6, y, ids, destinos, clima, ca, somas, contagens, cs = amostrar_arvores_seas5(chuva, atmosfera, seas, corte)\n    cc = ajustar_clima_previsao(cfs, corte, \'CFSv2\')\n    X = np.empty((len(x6), 31), dtype=np.float32)\n    X[:, :29] = x6\n    n = ids.shape[1]\n    for i, t in enumerate(destinos):\n        sl = slice(i * n, (i + 1) * n)\n        X[sl, 29] = valores_cfsv2(cfs, t, ids[i])\n        X[sl, 30] = valores_anomalia_cfsv2(cfs, cc, t, ids[i])\n    exigir(np.array_equal(X[:, :29], x6), \'29 features do controle alteradas.\')\n    exigir(np.isfinite(X).all() and np.isfinite(y).all(), \'Matriz de treino inválida.\')\n    return (X, y, ids, destinos, clima, ca, somas, contagens, cs, cc)\n\ndef prever_par(modelo, nome, atmosfera, seas, cfs, ca, clima, cs, cc, destinos):\n    destinos = pd.DatetimeIndex(destinos)\n    exigir(nome in NOMES_MODELOS, \'Modelo desconhecido.\')\n    exigir((destinos > pd.Timestamp(cs.attrs[\'corte\'])).all() and cs.attrs[\'corte\'] == cc.attrs[\'corte\'], \'Inferência fora do período posterior ao corte.\')\n    out = np.empty((len(destinos), clima.sizes[\'lat\'], clima.sizes[\'lon\']), dtype=np.float32)\n    for i, t in enumerate(destinos):\n        x = np.column_stack([matriz_mes(atmosfera, ca, clima, t), valores_seas5(seas, t), valores_anomalia_seas5(seas, cs, t)]).astype(np.float32)\n        if nome == \'arvores_multissistema\':\n            x = np.column_stack([x, valores_cfsv2(cfs, t), valores_anomalia_cfsv2(cfs, cc, t)]).astype(np.float32)\n        exigir(x.shape[1] == (29 if nome == \'arvores_seas5\' else 31) and np.isfinite(x).all(), \'Inferência inválida.\')\n        out[i] = modelo.predict(x).reshape(out.shape[1:])\n    exigir(np.isfinite(out).all(), \'Previsões inválidas.\')\n    return xr.DataArray(out, dims=(\'time\', \'lat\', \'lon\'), coords={\'time\': destinos, \'lat\': clima.lat, \'lon\': clima.lon}, attrs={\'units\': \'mm/day\'})\n\ndef mos_fit(chuva, seas, cfs, corte):\n    dates = calendario_pareado(corte)\n    r = chuva.sel(time=dates).transpose(\'time\', \'lat\', \'lon\').values.reshape(len(dates), -1)\n    s = seas.sel(time=dates).transpose(\'time\', \'lat\', \'lon\').values.reshape(len(dates), -1)\n    c = cfs.sel(time=dates).transpose(\'time\', \'lat\', \'lon\').values.reshape(len(dates), -1)\n    for pre in [seas, cfs]:\n        assert pd.DatetimeIndex(pre.sel(time=dates).time_origem.values).equals((dates.to_period(\'M\') - 1).to_timestamp())\n    assert (pd.DatetimeIndex(cfs.sel(time=dates).inicializacao_mais_recente.values) < dates).all()\n    baseline = estatisticas_climatologia(chuva, corte, 30)[0]\n    shape = baseline.shape[1:]\n    ng = r.shape[1]\n    coef = np.zeros((2, ng), np.float64)\n    mu = np.zeros((2, 12, ng), np.float64)\n    for lo in range(0, ng, 4096):\n        hi = min(lo + 4096, ng)\n        y = r[:, lo:hi].astype(np.float64)\n        x = s[:, lo:hi].astype(np.float64)\n        z = c[:, lo:hi].astype(np.float64)\n        assert np.isfinite(y).all() and np.isfinite(x).all() and np.isfinite(z).all()\n        for m in range(1, 13):\n            ix = dates.month == m\n            mu[0, m - 1, lo:hi] = x[ix].mean(0)\n            mu[1, m - 1, lo:hi] = z[ix].mean(0)\n            x[ix] -= mu[0, m - 1, lo:hi]\n            z[ix] -= mu[1, m - 1, lo:hi]\n            y[ix] -= y[ix].mean(0)\n        sx = np.sqrt(np.mean(x * x, axis=0))\n        sz = np.sqrt(np.mean(z * z, axis=0))\n        sx = np.where(sx > 1e-12, sx, 1.0)\n        sz = np.where(sz > 1e-12, sz, 1.0)\n        x /= sx\n        z /= sz\n        a = np.mean(x * x, axis=0) + MOS_ALPHA\n        b = np.mean(x * z, axis=0)\n        d = np.mean(z * z, axis=0) + MOS_ALPHA\n        u = np.mean(x * y, axis=0)\n        v = np.mean(z * y, axis=0)\n        det = a * d - b * b\n        assert (det > 0).all()\n        coef[0, lo:hi] = (d * u - b * v) / det / sx\n        coef[1, lo:hi] = (a * v - b * u) / det / sz\n    assert np.isfinite(coef).all()\n    return dict(coef=coef.reshape(2, *shape), mu=mu.reshape(2, 12, *shape), baseline=baseline.values, lat=baseline.lat.values, lon=baseline.lon.values, corte=str(pd.Timestamp(corte).date()), inicio=str(dates[0].date()), meses=len(dates))\n\ndef mos_predict(fit, seas, cfs, dates):\n    dates = pd.DatetimeIndex(dates)\n    assert (dates > pd.Timestamp(fit[\'corte\'])).all()\n    for pre in [seas, cfs]:\n        assert np.array_equal(pre.lat, fit[\'lat\']) and np.array_equal(pre.lon, fit[\'lon\'])\n    out = []\n    for t in dates:\n        s = valores_seas5(seas, t).reshape(fit[\'baseline\'].shape[1:]).astype(np.float64)\n        c = valores_cfsv2(cfs, t).reshape(s.shape).astype(np.float64)\n        anom = fit[\'coef\'][0] * (s - fit[\'mu\'][0, t.month - 1]) + fit[\'coef\'][1] * (c - fit[\'mu\'][1, t.month - 1])\n        out.append(np.maximum(0, fit[\'baseline\'][t.month - 1].astype(np.float64) + anom))\n    return np.stack(out)\n\ndef matriz_mes(campos, clima_atmos, clima_tp, destino, pontos=None):\n    """\n    Atmosfera T−4; índices T−3; meses reais preservados.\n\n    A climatologia da chuva é integral por padrão.\n    Somente amostrar_treino substitui sua coluna pelo leave-one-out.\n    """\n    destino = pd.Timestamp(destino)\n    origem = origem_mensal([destino], LAG_ATMOSFERA)[0]\n    nlat = clima_tp.sizes[\'lat\']\n    nlon = clima_tp.sizes[\'lon\']\n    pontos = np.arange(nlat * nlon) if pontos is None else np.asarray(pontos)\n    exigir(pontos.ndim == 1 and np.issubdtype(pontos.dtype, np.integer), \'Índices inválidos.\')\n    exigir(((pontos >= 0) & (pontos < nlat * nlon)).all(), \'Ponto fora da grade.\')\n    X = np.empty((len(pontos), len(FEATURES)), dtype=np.float32)\n    for j, nome in enumerate(VARIAVEIS):\n        campo = campos[nome]\n        i = pd.DatetimeIndex(campo.time.values).get_indexer([origem])[0]\n        exigir(i >= 0, f\'Origem {origem.date()} ausente em {nome}.\')\n        valores = campo.values[i].reshape(-1)[pontos]\n        clima_origem = clima_atmos[nome][origem.month - 1].reshape(-1)[pontos]\n        X[:, j] = valores\n        X[:, j + len(VARIAVEIS)] = valores - clima_origem\n    X[:, 18] = clima_tp.lat.values[pontos // nlon]\n    X[:, 19] = clima_tp.lon.values[pontos % nlon]\n    X[:, 20] = np.sin(2 * np.pi * destino.month / 12)\n    X[:, 21] = np.cos(2 * np.pi * destino.month / 12)\n    X[:, 22] = clima_tp.values[destino.month - 1].reshape(-1)[pontos]\n    exigir(origem < destino, \'Violação temporal: origem não é anterior ao alvo.\')\n    valores_oceano = valores_indices_mes(INDICES_OC, clima_atmos[\'_clima_indices\'], destino)\n    X[:, 23:] = valores_oceano[None, :]\n    exigir(np.isfinite(X).all(), \'Atributos contêm valores ausentes ou infinitos.\')\n    return X\n\nSST_COMPONENTES = 8\nSST_NOME = \'ersstv5_4graus_198201_202411.nc\'\nSST_HASH = \'908034f6418833302ea432f25850ad2982d1114b747b7b298e26a350eaf2891d\'\n\ndef sst_auditar_arquivo(path):\n    with xr.open_dataset(path) as ds:\n        exigir(\'sst\' in ds and ds.sst.dims==(\'time\',\'lat\',\'lon\'),\'SST: variável/dimensões incorretas.\')\n        exigir(ds.sst.attrs.get(\'units\')==\'degC\',\'SST deve estar em graus Celsius.\')\n        exigir(\'ERSSTv5\' in str(ds.attrs.get(\'title\',\'\')) and\n               str(ds.attrs.get(\'product_version\',\'\'))==\'Version 5\',\'Produto diferente de ERSSTv5.\')\n        t=pd.DatetimeIndex(ds.time.values)\n        exigir(t.equals(pd.date_range(\'1982-01-01\',\'2024-11-01\',freq=\'MS\')),\'SST: calendário incompleto ou diferente.\')\n        exigir(np.array_equal(ds.lat.values,np.arange(60,-61,-4)) and\n               np.array_equal(ds.lon.values,np.arange(0,360,4)),\'SST: esperado recorte 60S–60N em grade de 4 graus.\')\n        sst=ds.sst.astype(\'float32\').load()\n        atributos={k:str(v) for k,v in ds.attrs.items()}\n    v=sst.values;validos=v[np.isfinite(v)]\n    exigir(validos.size>0 and not np.isinf(v).any() and validos.min()>=-1.801 and validos.max()<=45,\n           \'SST: valores fora da faixa física do produto.\')\n    # Os NaNs terrestres serão excluídos por máscara ajustada apenas no treino de cada fold.\n    return sst,dict(arquivo=Path(path).name,sha256=sha256(path),bytes=Path(path).stat().st_size,\n        inicio_origens=str(t[0].date()),fim_origens=str(t[-1].date()),meses=len(t),grade=[31,90],\n        minimo=float(validos.min()),maximo=float(validos.max()),atributos=atributos,\n        referencia=\'https://psl.noaa.gov/data/gridded/data.noaa.ersst.v5.html\',\n        origem=\'NOAA PSL / ERSSTv5; snapshot retrospectivo, sem garantia de versão operacional histórica\')\n\ndef sst_ajustar_pca(sst,alvos_treino):\n    alvos_treino=pd.DatetimeIndex(alvos_treino)\n    exigir(alvos_treino.is_unique and alvos_treino.equals(pd.date_range(alvos_treino.min(),alvos_treino.max(),freq=\'MS\')),\n           \'Calendário de ajuste da PCA inválido.\')\n    origens=origem_mensal(alvos_treino, LAG_SST)\n    treino=sst.sel(time=origens).transpose(\'time\',\'lat\',\'lon\')\n    bruto=treino.values.reshape(len(origens),-1).astype(np.float64)\n    mascara=np.isfinite(bruto).all(axis=0)\n    exigir(mascara.any(),\'Nenhum ponto SST oceânico completo no treino.\')\n    mensal=np.full((12,bruto.shape[1]),np.nan,dtype=np.float64)\n    contagens=np.array([(origens.month==m).sum() for m in range(1,13)])\n    exigir((contagens>=2).all(),\'Climatologia SST mensal insuficiente.\')\n    for m in range(1,13):mensal[m-1,mascara]=bruto[origens.month==m][:,mascara].mean(axis=0)\n    anom=bruto[:,mascara]-mensal[origens.month-1][:,mascara]\n    ativos=np.flatnonzero(mascara)\n    mascara[ativos[np.std(anom,axis=0)<=1e-8]]=False\n    latitudes=np.repeat(sst.lat.values,sst.sizes[\'lon\'])\n    pesos=np.sqrt(np.cos(np.deg2rad(latitudes[mascara]))).astype(np.float64)\n    exigir(int(mascara.sum())>SST_COMPONENTES and len(origens)>SST_COMPONENTES,\'PCA sem dimensões suficientes.\')\n    matriz=(bruto[:,mascara]-mensal[origens.month-1][:,mascara])*pesos\n    exigir(np.isfinite(matriz).all(),\'Matriz de PCA inválida.\')\n    with threadpool_limits(limits=1):\n        pca=PCA(n_components=SST_COMPONENTES,svd_solver=\'full\').fit(matriz)\n    return dict(mascara=mascara,clima_mensal=mensal[:,mascara],pesos_area=pesos,\n        centro=pca.mean_,componentes=pca.components_,variancia_explicada=pca.explained_variance_ratio_,\n        contagens_mensais=contagens,origens_treino=origens.values,lat=sst.lat.values.copy(),lon=sst.lon.values.copy())\n\ndef sst_transformar(sst,estado,alvos):\n    alvos=pd.DatetimeIndex(alvos);origens=origem_mensal(alvos, LAG_SST)\n    exigir(np.array_equal(sst.lat,estado[\'lat\']) and np.array_equal(sst.lon,estado[\'lon\']),\'Grade SST mudou.\')\n    v=sst.sel(time=origens).values.reshape(len(origens),-1).astype(np.float64)[:,estado[\'mascara\']]\n    exigir(np.isfinite(v).all(),\'SST ausente no oceano selecionado pelo treino; não imputar usando futuro.\')\n    x=(v-estado[\'clima_mensal\'][origens.month-1])*estado[\'pesos_area\']-estado[\'centro\']\n    with threadpool_limits(limits=1):scores=x@estado[\'componentes\'].T\n    exigir(scores.shape==(len(alvos),SST_COMPONENTES) and np.isfinite(scores).all(),\'Scores PCA inválidos.\')\n    return scores.astype(np.float32)\n\ndef pesquisa_prever_sst(modelo, nome, atmosfera, seas, cfs, ca, clima, cs, cc, datas, scores):\n    datas = pd.DatetimeIndex(datas)\n    exigir((datas > pd.Timestamp(cs.attrs[\'corte\'])).all(), \'Inferência invade treino.\')\n    exigir(scores.shape == (len(datas), 8), \'Scores SST desalinhados.\')\n    out = np.empty((len(datas), clima.sizes[\'lat\'], clima.sizes[\'lon\']), dtype=np.float32)\n    expected = (FEATURES_SEAS5 if nome == \'arvores_seas5\' else FEATURES_MULTISSISTEMA) + SST_FEATURES\n    exigir(modelo.feature_name() == expected, \'Ordem de features SST incorreta.\')\n    for i, t in enumerate(datas):\n        x = np.column_stack([matriz_mes(atmosfera, ca, clima, t),\n            valores_seas5(seas, t), valores_anomalia_seas5(seas, cs, t)]).astype(np.float32)\n        if nome == \'arvores_multissistema\':\n            x = np.column_stack([x, valores_cfsv2(cfs, t), valores_anomalia_cfsv2(cfs, cc, t)]).astype(np.float32)\n        x = np.column_stack([x, np.broadcast_to(scores[i], (len(x), 8))]).astype(np.float32)\n        exigir(x.shape[1] == len(expected) and np.isfinite(x).all(), \'Features SST inválidas.\')\n        out[i] = modelo.predict(x).reshape(out.shape[1:])\n    return xr.DataArray(out, dims=(\'time\', \'lat\', \'lon\'),\n        coords=dict(time=datas, lat=clima.lat, lon=clima.lon), attrs=dict(units=\'mm/day\'))\n\ndef pesquisa_diagnosticos(obs, previsoes, bloco):\n    """Dados completos, sem amostragem de pixels e sem mascarar erros."""\n    linhas = []\n    lat = obs.lat.values\n    regioes = {\'dominio_inteiro\': np.ones(lat.size, dtype=bool),\n        \'lat_menor_que_menos35\': lat < -35,\n        \'lat_menos35_a_menos15\': (lat >= -35) & (lat < -15),\n        \'lat_maior_igual_menos15\': lat >= -15}\n    for modelo, pred in previsoes.items():\n        obs, pred = xr.align(obs, pred, join=\'exact\')\n        for i, t in enumerate(pd.DatetimeIndex(obs.time.values)):\n            erro = pred.values[i].astype(np.float64) - obs.values[i].astype(np.float64)\n            exigir(np.isfinite(erro).all(), \'Erro não finito nos diagnósticos.\')\n            for regiao, mask in regioes.items():\n                if not mask.any():\n                    continue\n                e = erro[mask]\n                w = np.broadcast_to(np.cos(np.deg2rad(lat[mask]))[:, None], e.shape)\n                linhas.append(dict(modelo=modelo, bloco=bloco, mes_alvo=str(t.date()),\n                    ano=t.year, mes=t.month, regiao=regiao, n=e.size,\n                    sse=float(np.square(e).sum()), soma_erro=float(e.sum()),\n                    soma_erro_absoluto=float(np.abs(e).sum()),\n                    sse_area=float((w * np.square(e)).sum()), peso_area=float(w.sum())))\n    return pd.DataFrame(linhas)\n\ndef pesquisa_resumir(mensais, grupos):\n    cols = [\'n\', \'sse\', \'soma_erro\', \'soma_erro_absoluto\', \'sse_area\', \'peso_area\']\n    r = mensais.groupby(grupos, as_index=False)[cols].sum()\n    r[\'rmse\'] = np.sqrt(r.sse / r.n)\n    r[\'mae\'] = r.soma_erro_absoluto / r.n\n    r[\'vies\'] = r.soma_erro / r.n\n    r[\'rmse_area\'] = np.sqrt(r.sse_area / r.peso_area)\n    return r\n\n"""Política temporal candidata; não certifica vintages nem datas de publicação."""\nLAG_ATMOSFERA = 4\nLAG_INDICES = 3\nLAG_SST = 2\nLAG_CHUVA_TREINO = 4\nINICIO_COMUM = pd.Timestamp(\'1982-03-01\')\nFEATURES = ([v + \'_lag4\' for v in VARIAVEIS]\n    + [v + \'_lag4_anomalia\' for v in VARIAVEIS]\n    + [\'latitude\', \'longitude\', \'mes_alvo_sin\', \'mes_alvo_cos\', \'clima_tp_alvo\']\n    + [n + \'_lag3_anomalia_fold\' for n in INDICES_NOMES])\nFEATURES_SEAS5 = FEATURES + [FEATURE_SEAS5, FEATURE_SEAS5_ANOM]\nFEATURES_MULTISSISTEMA = FEATURES_SEAS5 + [FEATURE_CFSV2, FEATURE_CFSV2_ANOM]\nSST_FEATURES = [f\'sst_lag2_pc_{i:02d}_treino_fold\' for i in range(1, 9)]\n\n\ndef origem_mensal(alvos, lag):\n    datas = pd.DatetimeIndex(alvos)\n    exigir(lag >= 1 and datas.equals(datas.to_period(\'M\').to_timestamp()), \'Datas/lag inválidos.\')\n    return (datas.to_period(\'M\') - lag).to_timestamp()\n\n\ndef janela_referencia(corte, anos=30):\n    corte = pd.Timestamp(corte)\n    exigir(corte.day == 1, \'Corte precisa identificar um mês.\')\n    return pd.date_range(end=corte, periods=12 * anos, freq=\'MS\')\n\n\ndef calendario_pareado(corte):\n    datas = janela_referencia(corte)\n    datas = datas[datas >= INICIO_COMUM]\n    exigir(24 <= len(datas) <= 360, \'Janela comum inválida.\')\n    return datas\n\n\ndef calendario_pares(datas_disponiveis, corte, anos=30):\n    alvos = janela_referencia(corte, anos)\n    origens = origem_mensal(alvos, LAG_ATMOSFERA)\n    disponiveis = pd.DatetimeIndex(datas_disponiveis)\n    exigir(disponiveis.is_unique and origens.isin(disponiveis).all(), \'Faltam origens atmosféricas.\')\n    return origens, alvos\n\n\ndef estatisticas_climatologia(tp, corte, anos=30):\n    datas = janela_referencia(corte, anos)\n    treino = tp.sel(time=datas).transpose(\'time\', \'lat\', \'lon\')\n    exigir(np.isfinite(treino.values).all(), \'Chuva de treino incompleta.\')\n    somas = np.stack([treino.values[datas.month == m].sum(axis=0, dtype=np.float64)\n                      for m in range(1, 13)])\n    contagens = np.array([(datas.month == m).sum() for m in range(1, 13)], dtype=np.int64)\n    exigir((contagens == anos).all(), \'Climatologia precisa de todos os meses.\')\n    clima = xr.DataArray((somas / contagens[:, None, None]).astype(np.float32),\n        dims=(\'month\', \'lat\', \'lon\'), coords=dict(month=np.arange(1, 13), lat=tp.lat, lon=tp.lon),\n        attrs=dict(units=\'mm/day\', ajuste_inicio=str(datas[0].date()), ajuste_fim=str(datas[-1].date())))\n    return clima, somas, contagens\n\n\ndef ajustar_clima_indices(tabela, origens_atmosfera, corte):\n    # Calendário próprio: índices T−3, enquanto atmosfera utiliza T−4.\n    alvos = janela_referencia(corte)\n    exigir(pd.DatetimeIndex(origens_atmosfera).equals(origem_mensal(alvos, LAG_ATMOSFERA)),\n           \'Referência atmosférica fora do protocolo.\')\n    origens = origem_mensal(alvos, LAG_INDICES)\n    valores = tabela.loc[origens, INDICES_NOMES].to_numpy(dtype=np.float64)\n    exigir(np.isfinite(valores).all(), \'Índices de treino incompletos.\')\n    return np.stack([valores[origens.month == m].mean(axis=0, dtype=np.float64) for m in range(1, 13)])\n\n\ndef valores_indices_mes(tabela, clima_indices, destino):\n    origem = origem_mensal([destino], LAG_INDICES)[0]\n    valores = tabela.loc[origem, INDICES_NOMES].to_numpy(dtype=np.float64)\n    exigir(np.isfinite(valores).all(), \'Índice indisponível; não preencher.\')\n    return (valores - clima_indices[origem.month - 1]).astype(np.float32)\n\n\ndef exigir_disponibilidade_real(registros, emitido_em, fontes_necessarias):\n    """Porta para futura produção; não é usada para certificar este retrospectivo.\n\n    O registro precisa se referir aos bytes recebidos, não à data nominal da previsão.\n    Valida metadados fornecidos; sua autenticidade exige um coletor auditável.\n    """\n    emissao = pd.Timestamp(emitido_em)\n    exigir(emissao.tzinfo is not None, \'Emissão exige timezone.\')\n    exigir(len(registros) == len(fontes_necessarias), \'Quantidade de fontes incorreta.\')\n    exigir({r.get(\'fonte\') for r in registros} == set(fontes_necessarias), \'Fonte faltante/duplicada.\')\n    for r in registros:\n        for chave in [\'disponivel_em\', \'recebido_em\', \'url\', \'sha256\', \'versao\']:\n            exigir(bool(r.get(chave)), \'Disponibilidade desconhecida: \' + r.get(\'fonte\', \'?\') + \'/\' + chave)\n        recebido, publicado = pd.Timestamp(r[\'recebido_em\']), pd.Timestamp(r[\'disponivel_em\'])\n        exigir(recebido.tzinfo is not None and publicado.tzinfo is not None, \'Datas exigem timezone.\')\n        exigir(publicado <= recebido <= emissao, \'Fonte chegou após a emissão ou cronologia inválida.\')\n        exigir(len(r[\'sha256\']) == 64 and set(r[\'sha256\']) <= set(\'0123456789abcdef\'), \'SHA-256 inválido.\')\n    return True\n\n\ndef calendario_emissoes(datas, corte):\n    datas = pd.DatetimeIndex(datas)\n    limite = origem_mensal([datas[0]], LAG_CHUVA_TREINO)[0]\n    exigir(pd.Timestamp(corte) <= limite, \'Chuva de treino recente demais para a primeira emissão.\')\n    return pd.DataFrame(dict(time_alvo=datas, limite_emissao=datas - pd.Timedelta(nanoseconds=1),\n        atmosfera=origem_mensal(datas, LAG_ATMOSFERA), indices=origem_mensal(datas, LAG_INDICES),\n        sst=origem_mensal(datas, LAG_SST), inicializacao_sazonal=origem_mensal(datas, 1),\n        ultimo_alvo_treino=pd.Timestamp(corte), disponibilidade_historica_comprovada=False))\n\n\ndef auditar_invariancia_defasada(chuva, atmosfera, seas, cfs, sst, corte, datas, clima, ca, cs, cc, pca):\n    """Muta cópias pequenas reais; dados após cada limite não podem afetar T."""\n    primeiro = pd.Timestamp(datas[0])\n    pontos = np.array([0, 1, chuva.sizes[\'lon\']], dtype=np.int32)\n    # Recorte espacial pequeno mantém todas as datas para testar os ajustes.\n    ch = chuva.isel(lat=slice(0, 2), lon=slice(0, 2)).copy(deep=True)\n    at = {k: v.isel(lat=slice(0, 2), lon=slice(0, 2)).copy(deep=True) for k, v in atmosfera.items()}\n    alvos_ref = janela_referencia(corte)\n    orig_at = origem_mensal(alvos_ref, LAG_ATMOSFERA)\n    c0, _, _ = estatisticas_climatologia(ch, corte)\n    a0 = ajustar_clima_atmosfera(at, orig_at)\n    a0[\'_clima_indices\'] = ajustar_clima_indices(INDICES_OC, orig_at, corte)\n    x0 = matriz_mes(at, a0, c0, primeiro)\n    ch.values[pd.DatetimeIndex(ch.time.values) > pd.Timestamp(corte)] = 123456\n    c1, _, _ = estatisticas_climatologia(ch, corte)\n    exigir(np.array_equal(c0.values, c1.values), \'Chuva após corte alterou climatologia.\')\n    for v in at.values():\n        v.values[pd.DatetimeIndex(v.time.values) > origem_mensal([primeiro], LAG_ATMOSFERA)[0]] = 7654321\n    a1 = ajustar_clima_atmosfera(at, orig_at)\n    a1[\'_clima_indices\'] = a0[\'_clima_indices\']\n    exigir(all(np.array_equal(a0[k], a1[k]) for k in a0), \'Futuro alterou climatologia atmosférica.\')\n    exigir(np.array_equal(x0, matriz_mes(at, a1, c1, primeiro)), \'Atmosfera recente/futura alterou T.\')\n    ni = INDICES_OC.copy(deep=True)\n    ni.loc[ni.index > origem_mensal([primeiro], LAG_INDICES)[0], :] = 99999\n    ci = ajustar_clima_indices(ni, orig_at, corte)\n    exigir(np.array_equal(ci, a0[\'_clima_indices\']), \'Índices futuros alteraram ajuste.\')\n    exigir(np.array_equal(valores_indices_mes(ni, ci, primeiro), x0[0, 23:]), \'Índice futuro alterou feature.\')\n    for pre, nome in [(seas, \'SEAS5\'), (cfs, \'CFSv2\')]:\n        pequeno = pre.isel(lat=slice(0, 2), lon=slice(0, 2)).copy(deep=True)\n        antes = ajustar_clima_previsao(pequeno, corte, nome)\n        pequeno.values[pd.DatetimeIndex(pequeno.time.values) > pd.Timestamp(corte)] = 12345\n        exigir(np.array_equal(antes, ajustar_clima_previsao(pequeno, corte, nome)), \'Futuro alterou clima sazonal.\')\n    copia = sst.copy(deep=True)\n    copia.values[pd.DatetimeIndex(copia.time.values) > origem_mensal([primeiro], LAG_SST)[0]] = 12345\n    novo = sst_ajustar_pca(copia, calendario_pareado(corte))\n    for k in [\'mascara\', \'clima_mensal\', \'centro\', \'componentes\', \'origens_treino\']:\n        exigir(np.array_equal(novo[k], pca[k]), \'SST futura alterou PCA: \' + k)\n    exigir(np.array_equal(sst_transformar(copia, novo, [primeiro]), sst_transformar(sst, pca, [primeiro])),\n           \'SST mais recente que T−2 alterou previsão de T.\')\n    return dict(chuva_pos_corte_invariante=True, atmosfera_apos_Tmenos4_invariante=True,\n        indices_apos_Tmenos3_invariantes=True, sst_apos_Tmenos2_invariante=True,\n        climas_sazonais_somente_treino=True, publicacao_historica_comprovada=False)\n\n', 'executar_comparacao.py': '"""Experimento fixo de defasagens, em sete blocos históricos já consultados."""\nfrom pathlib import Path\nimport runpy\n\nCODIGO = Path(__file__).resolve().parent\nRAIZ = CODIGO.parent\nexec(compile((CODIGO/\'biblioteca.py\').read_text(encoding=\'utf-8\'), str(CODIGO/\'biblioteca.py\'), \'exec\'), globals())\nimport zipfile\nimport matplotlib.pyplot as plt\n\n\ndef registrar(path, obj):\n    Path(path).write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str), encoding=\'utf-8\')\n\n\ndef arr_hash(a):\n    return hashlib.sha256(np.ascontiguousarray(a).tobytes()).hexdigest()\n\n\ndef conferir_configuracao(protocolo):\n    exigir((LAG_ATMOSFERA, LAG_INDICES, LAG_SST, LAG_CHUVA_TREINO) == (4, 3, 2, 4), \'Defasagens alteradas.\')\n    exigir((ARVORES, SEMENTE, PONTOS_POR_MES, ANOS_TREINO, MOS_ALPHA) == (300, 42, 5000, 30, .1), \'Configuração alterada.\')\n    exigir(protocolo[\'pesos\'] == [.375, .375, .25] and protocolo[\'escalas_residuais\'] == [.9, .875], \'Pesos alterados.\')\n    exigir(protocolo[\'consultar_2025\'] is False and protocolo[\'avaliacao\'] == [\'2007-01-01\', \'2020-12-01\'], \'Avaliação alterada.\')\n    esperado = {\'objective\':\'regression\',\'metric\':\'rmse\',\'learning_rate\':.05,\'num_leaves\':31,\n        \'min_data_in_leaf\':150,\'lambda_l2\':10.,\'feature_fraction\':.9,\'bagging_fraction\':.8,\n        \'bagging_freq\':1,\'num_threads\':4,\'seed\':42,\'deterministic\':True,\'force_col_wise\':True,\'verbosity\':-1}\n    exigir(PARAMETROS == esperado, \'Parâmetros diferentes da comparação fixada.\')\n\n\ndef resultado_final(detalhes):\n    exigir(not detalhes.duplicated([\'modelo\',\'mes_alvo\',\'regiao\']).any(), \'Meses duplicados.\')\n    exigir(set(detalhes.modelo) == {\'controle_defasado\',\'sst_defasada\',\'climatologia\'}, \'Tratamentos incompletos.\')\n    datas = pd.DatetimeIndex(pd.to_datetime(detalhes.mes_alvo.unique())).sort_values()\n    exigir(datas.equals(pd.date_range(\'2007-01-01\',\'2020-12-01\',freq=\'MS\')), \'Período incompleto.\')\n    dominio = detalhes[detalhes.regiao == \'dominio_inteiro\']\n    globais = pesquisa_resumir(dominio, [\'modelo\'])\n    exigir((globais.n == 13198248).all(), \'Quantidade de previsões inesperada.\')\n    blocos = pesquisa_resumir(dominio, [\'modelo\',\'bloco\'])\n    anos = pesquisa_resumir(dominio, [\'modelo\',\'ano\'])\n    regioes = pesquisa_resumir(detalhes, [\'modelo\',\'regiao\'])\n    for nome, df in [(\'mensais_regioes\',detalhes),(\'globais\',globais),(\'blocos\',blocos),(\'anos\',anos),(\'regioes\',regioes)]:\n        df.to_csv(RAIZ/f\'metricas_{nome}.csv\',index=False,lineterminator=\'\\n\')\n    consistencia = {}\n    for nome, df, eixo in [(\'blocos\',blocos,\'bloco\'),(\'anos\',anos,\'ano\')]:\n        p = df.pivot(index=eixo,columns=\'modelo\',values=\'rmse\')\n        delta = p.sst_defasada - p.controle_defasado\n        consistencia[nome] = dict(melhores=int((delta<0).sum()), piores=int((delta>0).sum()),\n                                  empates=int((delta==0).sum()), pior_delta=float(delta.max()), melhor_delta=float(delta.min()))\n    g = globais.set_index(\'modelo\')\n    delta = float(g.loc[\'sst_defasada\',\'rmse\'] - g.loc[\'controle_defasado\',\'rmse\'])\n    resumo = dict(status=\'completo\', protocolo_sha256=sha256(CODIGO/\'protocolo.json\'),\n        rmse_controle=float(g.loc[\'controle_defasado\',\'rmse\']), rmse_sst=float(g.loc[\'sst_defasada\',\'rmse\']),\n        delta_rmse=delta, reducao_percentual=-100*delta/float(g.loc[\'controle_defasado\',\'rmse\']),\n        consistencia=consistencia, disponibilidade_historica_comprovada=False,\n        holdout_independente=False, producao_liberada=False, alvos_2025_lidos=False)\n    registrar(RAIZ/\'resumo.json\',resumo)\n    linhas = [\'| Modelo | RMSE | MAE | Viés | RMSE por área |\',\'|---|---:|---:|---:|---:|\']\n    labels = {\'controle_defasado\':\'Híbrido com defasagens\',\'sst_defasada\':\'Híbrido com defasagens + 8 PCs SST\',\'climatologia\':\'Climatologia\'}\n    for n in labels:\n        r = g.loc[n]\n        linhas.append(f\'| {labels[n]} | {r.rmse:.6f} | {r.mae:.6f} | {r.vies:+.6f} | {r.rmse_area:.6f} |\')\n    rel = f\'\'\'# Nimbus PA — comparação com fontes defasadas\n\nAvaliação 2007–2020, em 13.198.248 pontos-meses por modelo. Sete blocos históricos;\nambos os híbridos foram treinados novamente com os mesmos exemplos e limites temporais.\n\n{chr(10).join(linhas)}\n\nUnidades: mm/dia. **Delta de RMSE SST menos controle: {delta:+.6f}**.\nSST melhora em **{consistencia[\'blocos\'][\'melhores\']}/7 blocos** e **{consistencia[\'anos\'][\'melhores\']}/14 anos**.\nAs métricas globais usam somas de erros, não médias de RMSEs.\n\nAtmosfera ERA5 T−4, índices T−3, SST T−2, previsões sazonais inicializadas em T−1.\nO treino de cada bloco termina em setembro anterior ao primeiro janeiro previsto;\nhá no máximo 360 meses. Árvores, ridge, pesos e escalas ficam fixos. A PCA é ajustada\nsomente às SSTs associadas aos meses de treinamento e não recebe chuva como entrada.\n\n**Interpretação:** sensibilidade retrospectiva em anos já usados no desenvolvimento.\nArquivos consolidados e hindcasts não reconstituem as versões publicadas em cada data.\nA chegada do agregado CFSv2/IRI continua sem calendário histórico comprovado. Por isso,\nesta execução não libera previsões operacionais e não garante melhora futura.\n2025 não foi carregado, treinado ou reavaliado nesta etapa.\n\nCada bloco conserva modelos, PCA, climatologias, regressão, previsões e auditorias.\nO protocolo foi registrado antes do primeiro ajuste desta comparação; não houve busca de lags ou pesos.\n\'\'\'\n    (RAIZ/\'resultado.md\').write_text(rel,encoding=\'utf-8\')\n    anual = anos.pivot(index=\'ano\',columns=\'modelo\',values=\'rmse\')\n    dp = anual.sst_defasada-anual.controle_defasado\n    with plt.rc_context({\'axes.spines.top\':False,\'axes.spines.right\':False}):\n        fig, ax = plt.subplots(2,1,figsize=(11,7),layout=\'constrained\')\n        for n,col in [(\'controle_defasado\',\'#566980\'),(\'sst_defasada\',\'#087f79\')]:\n            ax[0].plot(anual.index,anual[n],marker=\'o\',label=labels[n],color=col)\n        ax[0].set(title=\'Mesmas fontes defasadas, com e sem SST global\',ylabel=\'RMSE (mm/dia)\')\n        ax[0].legend(frameon=False)\n        ax[1].bar(dp.index,dp,color=[\'#087f79\' if v<0 else \'#bc654b\' for v in dp])\n        ax[1].axhline(0,color=\'#253a47\',lw=.8)\n        ax[1].set(title=\'Contribuição da SST por ano — valores negativos indicam melhora\',ylabel=\'Delta RMSE (mm/dia)\',xticks=dp.index,xlabel=\'Ano\')\n        for a in ax:a.grid(axis=\'y\',alpha=.15)\n        fig.savefig(RAIZ/\'comparacao.png\',dpi=160);plt.close(fig)\n    itens = []\n    for p in sorted(RAIZ.rglob(\'*\')):\n        if p.is_file() and p.suffix not in {\'.zip\',\'.pyc\'} and \'__pycache__\' not in p.parts:\n            itens.append(dict(arquivo=p.relative_to(RAIZ).as_posix(),sha256=sha256(p),bytes=p.stat().st_size))\n    registrar(RAIZ/\'inventario.json\',itens)\n    with zipfile.ZipFile(RAIZ/\'relatorios.zip\',\'w\',zipfile.ZIP_DEFLATED) as z:\n        for r in itens:\n            p = RAIZ/r[\'arquivo\']\n            if p.suffix in {\'.json\',\'.csv\',\'.md\',\'.png\',\'.py\'}: z.write(p,r[\'arquivo\'])\n        z.write(RAIZ/\'inventario.json\',\'inventario.json\')\n    print(\'COMPARAÇÃO CONCLUÍDA\',json.dumps(resumo,ensure_ascii=False),flush=True)\n    display(globais[[\'modelo\',\'rmse\',\'mae\',\'vies\',\'rmse_area\']])\n    return resumo\n\n\ndef executar():\n    global PASTA, PASTA_SEAS5, MANIFESTO_SEAS5, PASTA_CFSV2, MANIFESTO_CFSV2, INDICES_OC, SAIDA\n    SAIDA = RAIZ\n    protocolo = json.loads((CODIGO/\'protocolo.json\').read_text(encoding=\'utf-8\'))\n    conferir_configuracao(protocolo)\n    versoes = {n:importlib.metadata.version(n) for n in [\'numpy\',\'pandas\',\'xarray\',\'lightgbm\',\'scikit-learn\',\'scipy\']}\n    exigir(all(versoes[k] == v for k,v in {\'numpy\':\'2.0.2\',\'pandas\':\'2.3.3\',\'xarray\':\'2025.12.0\',\'lightgbm\':\'4.6.0\'}.items()),\n           \'Use o ambiente da referência; versões atuais: \'+str(versoes))\n    PASTA = localizar_dados(None)\n    PASTA_SEAS5, MANIFESTO_SEAS5 = localizar_seas5()\n    PASTA_CFSV2, MANIFESTO_CFSV2 = localizar_cfsv2()\n    entradas = []\n    for r in json.loads((CODIGO/\'hashes_oficiais.json\').read_text()):\n        if not r[\'nome\'].startswith(\'treino_\'): continue\n        p = PASTA/r[\'nome\']\n        exigir(p.is_file() and sha256(p)==r[\'sha256\'],\'Arquivo oficial alterado: \'+r[\'nome\'])\n        entradas.append(r)\n    ssts = sorted(Path(\'/kaggle/input\').rglob(SST_NOME))\n    exigir(ssts and all(sha256(p)==SST_HASH for p in ssts),\'Snapshot SST ausente ou alterado.\')\n    sst, meta_sst = sst_auditar_arquivo(ssts[0])\n    sst = sst.sel(time=slice(None,\'2020-10-01\'))\n    assinatura = dict(protocolo_sha256=sha256(CODIGO/\'protocolo.json\'),\n        codigo={p.name:sha256(p) for p in sorted(CODIGO.iterdir()) if p.is_file()},\n        entradas=entradas, versoes=versoes, sst_sha256=SST_HASH,\n        manifesto_seas5=sha256(PASTA_SEAS5/\'seas5_51_manifesto.json\'),\n        manifesto_cfsv2=sha256(PASTA_CFSV2/\'cfsv2_manifesto.json\'))\n    if (RAIZ/\'assinatura_execucao.json\').exists():\n        exigir(json.loads((RAIZ/\'assinatura_execucao.json\').read_text())==assinatura,\'Código, fontes ou ambiente mudaram; não retomar nesta pasta.\')\n    else: registrar(RAIZ/\'assinatura_execucao.json\',assinatura)\n    registrar(RAIZ/\'sst_fonte.json\',meta_sst)\n    print(\'Nimbus PA — defasagens 4/3/2; treino termina em T−4. Fontes e protocolo conferidos.\',flush=True)\n    chuva = auditar_chuva_desenv()\n    atmosfera, meta_at = carregar_atmosfera(chuva,pd.Timestamp(\'1976-06-01\'),pd.Timestamp(\'2020-08-01\'))\n    INDICES_OC = pd.read_csv(CODIGO/\'indices_noaa_197607_202009.csv\',parse_dates=[\'time_origem\']).set_index(\'time_origem\')\n    conferir_indices(INDICES_OC)\n    seas = carregar_seas5(\'desenvolvimento\',chuva)\n    cfs = carregar_cfsv2(\'desenvolvimento\',chuva)\n    registrar(RAIZ/\'atmosfera_fontes.json\',meta_at)\n    todos = []\n    for b in BLOCOS:\n        inicio = pd.Timestamp(b[\'inicio\'])\n        corte = origem_mensal([inicio],LAG_CHUVA_TREINO)[0]\n        datas = pd.date_range(inicio,periods=24,freq=\'MS\')\n        pasta = RAIZ/b[\'nome\'];pasta.mkdir(exist_ok=True)\n        if (pasta/\'concluido.json\').exists():\n            check = json.loads((pasta/\'concluido.json\').read_text())\n            exigir(check[\'assinatura\']==sha256(RAIZ/\'assinatura_execucao.json\'),\'Assinatura do bloco mudou.\')\n            for nome,h in check[\'arquivos\'].items(): exigir(sha256(pasta/nome)==h,\'Cache alterado: \'+nome)\n            todos.append(pd.read_csv(pasta/\'metricas.csv\'))\n            print(b[\'nome\'],\'cache conferido.\',flush=True);continue\n        calendario_emissoes(datas,corte).to_csv(pasta/\'calendario_emissoes.csv\',index=False)\n        # A função de treinamento só recebe chuva até o corte, nem mesmo acessa meses recentes.\n        chuva_treino = chuva.sel(time=slice(None,corte))\n        X,y,ids,alvos,clima,ca,somas,contagens,cs,cc = amostrar_arvores_multissistema(chuva_treino,atmosfera,seas,cfs,corte)\n        exigir(alvos[-1]==corte and alvos.max()<=origem_mensal([datas[0]],4)[0],\'Rótulos posteriores ao limite.\')\n        pca = sst_ajustar_pca(sst,alvos)\n        pcs = sst_transformar(sst,pca,alvos)\n        pcv = sst_transformar(sst,pca,datas)\n        auditoria = auditar_invariancia_defasada(chuva,atmosfera,seas,cfs,sst,corte,datas,clima,ca,cs,cc,pca)\n        auditoria.update(inicio=str(alvos[0].date()),corte=str(corte.date()),meses=len(alvos),\n            n_exemplos=len(y),sha256_X=arr_hash(X),sha256_y=arr_hash(y),sha256_ids=arr_hash(ids),\n            identidade_pareada=\'mesmas colunas base, mesmos y/ids/referências/ridge; só acrescenta 8 PCs\',\n            referencia_chuva=[str(janela_referencia(corte)[0].date()),str(corte.date())],\n            fontes_clima=dict(atmosfera=[str(origem_mensal(janela_referencia(corte),4)[0].date()),str(origem_mensal(janela_referencia(corte),4)[-1].date())],\n                              indices=[str(origem_mensal(janela_referencia(corte),3)[0].date()),str(origem_mensal(janela_referencia(corte),3)[-1].date())]))\n        registrar(pasta/\'auditoria.json\',auditoria)\n        np.savez_compressed(pasta/\'pca.npz\',**pca)\n        np.savez_compressed(pasta/\'amostras.npz\',pontos=ids,time_alvo=alvos.values)\n        np.savez_compressed(pasta/\'clima_atmosfera_indices.npz\',**ca)\n        for nome,cl in [(\'chuva\',clima),(\'seas5\',cs),(\'cfsv2\',cc)]:cl.to_netcdf(pasta/f\'clima_{nome}.nc\')\n        fit = mos_fit(chuva_treino,seas,cfs,corte)\n        np.savez_compressed(pasta/\'ridge.npz\',**fit)\n        ridge = mos_predict(fit,seas,cfs,datas)\n        modelos, previsoes = {}, {}\n        print(f"{b[\'nome\']} | treino {alvos[0].date()} a {corte.date()} | {len(y):,} exemplos; quatro árvores e ridge compartilhado",flush=True)\n        for tratamento in [\'controle_defasado\',\'sst_defasada\']:\n            componentes = []\n            for nome,base_features,scale in [(\'arvores_seas5\',FEATURES_SEAS5,.9),(\'arvores_multissistema\',FEATURES_MULTISSISTEMA,.875)]:\n                t0 = perf_counter()\n                nbase = len(base_features)\n                entrada = np.ascontiguousarray(X[:,:nbase])\n                features = list(base_features)\n                if tratamento==\'sst_defasada\':\n                    entrada = np.column_stack([entrada,np.repeat(pcs,ids.shape[1],axis=0)]).astype(np.float32)\n                    features += SST_FEATURES\n                exigir(np.array_equal(entrada[:,:nbase],X[:,:nbase]),\'Matriz base alterada.\')\n                data = lgb.Dataset(entrada,label=y,feature_name=features)\n                model = lgb.train(PARAMETROS,data,num_boost_round=ARVORES)\n                exigir(model.feature_name()==features,\'Ordem de features alterada.\')\n                model.save_model(str(pasta/f\'{tratamento}_{nome}.txt\'))\n                del data,entrada;gc.collect()\n                if tratamento==\'sst_defasada\':\n                    anom = pesquisa_prever_sst(model,nome,atmosfera,seas,cfs,ca,clima,cs,cc,datas,pcv)\n                else: anom = prever_par(model,nome,atmosfera,seas,cfs,ca,clima,cs,cc,datas)\n                componentes.append(reconstruir_chuva(clima,anom,scale).values.astype(np.float64))\n                print(b[\'nome\'],tratamento,nome,f\'ajuste e previsão: {perf_counter()-t0:.1f}s\',flush=True)\n                del model,anom\n            previsoes[tratamento] = .75*(.5*componentes[0]+.5*componentes[1])+.25*ridge\n            del componentes\n        exigir(arr_hash(X)==auditoria[\'sha256_X\'] and arr_hash(y)==auditoria[\'sha256_y\'],\'Treino foi modificado.\')\n        previsoes[\'climatologia\'] = clima.values[datas.month-1].astype(np.float64)\n        coords=dict(time=datas,lat=chuva.lat,lon=chuva.lon)\n        preds = {n:xr.DataArray(v,dims=(\'time\',\'lat\',\'lon\'),coords=coords,attrs=dict(units=\'mm/day\')) for n,v in previsoes.items()}\n        ds = xr.Dataset(preds,attrs=dict(natureza=\'retrospectiva; vintages não comprovados\',corte_treino=str(corte.date())))\n        ds.to_netcdf(pasta/\'previsoes.nc\',engine=\'h5netcdf\',encoding={n:dict(zlib=True,complevel=4) for n in ds.data_vars})\n        # Verificação fora da função de treinamento; nenhuma métrica controla o ajuste.\n        metricas = pesquisa_diagnosticos(chuva.sel(time=datas),preds,b[\'nome\'])\n        metricas.to_csv(pasta/\'metricas.csv\',index=False,lineterminator=\'\\n\')\n        arquivos = {p.name:sha256(p) for p in sorted(pasta.iterdir()) if p.is_file() and p.name!=\'concluido.json\'}\n        registrar(pasta/\'concluido.json\',dict(assinatura=sha256(RAIZ/\'assinatura_execucao.json\'),arquivos=arquivos))\n        todos.append(metricas)\n        bloco = pesquisa_resumir(metricas[metricas.regiao==\'dominio_inteiro\'],[\'modelo\'])\n        print(b[\'nome\'],\'concluído:\',bloco[[\'modelo\',\'rmse\']].to_dict(\'records\'),flush=True)\n        del X,y,ids,clima,ca,somas,contagens,cs,cc,pca,pcs,pcv,fit,ridge,preds,ds,previsoes;gc.collect()\n    return resultado_final(pd.concat(todos,ignore_index=True))\n\n\nif __name__==\'__main__\':\n    RESULTADO = executar()\n', 'testar_contrato.py': '"""Testes de calendário e causalidade; não treinam árvores com dados reais."""\nfrom pathlib import Path\nimport runpy\nimport unittest\nimport json\n\nB = runpy.run_path(str(Path(__file__).resolve().parent/\'biblioteca.py\'))\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\n\n\nclass ContratoTemporal(unittest.TestCase):\n    def test_virada_ano_e_janela(self):\n        self.assertEqual(B[\'origem_mensal\']([\'2007-01-01\'],4)[0],pd.Timestamp(\'2006-09-01\'))\n        self.assertEqual(B[\'origem_mensal\']([\'2007-01-01\'],3)[0],pd.Timestamp(\'2006-10-01\'))\n        self.assertEqual(B[\'origem_mensal\']([\'2007-01-01\'],2)[0],pd.Timestamp(\'2006-11-01\'))\n        j=B[\'janela_referencia\'](\'2006-09-01\')\n        self.assertEqual(j[0],pd.Timestamp(\'1976-10-01\'))\n        self.assertEqual(len(j),360)\n        self.assertTrue(all((j.month==m).sum()==30 for m in range(1,13)))\n        self.assertEqual(B[\'calendario_pareado\'](\'2006-09-01\')[0],pd.Timestamp(\'1982-03-01\'))\n\n    def test_embargo_chuva(self):\n        good=B[\'calendario_emissoes\'](pd.date_range(\'2007-01-01\',periods=24,freq=\'MS\'),\'2006-09-01\')\n        self.assertTrue((good.ultimo_alvo_treino<good.time_alvo).all())\n        with self.assertRaises(ValueError):\n            B[\'calendario_emissoes\']([\'2007-01-01\'],\'2006-12-01\')\n\n    def test_clima_chuva_sem_futuro(self):\n        ts=pd.date_range(\'1976-10-01\',\'2008-12-01\',freq=\'MS\')\n        a=xr.DataArray(np.arange(len(ts)*4,dtype=np.float32).reshape(-1,2,2),\n            dims=(\'time\',\'lat\',\'lon\'),coords=dict(time=ts,lat=[-40.,0.],lon=[-80.,-50.]))\n        cl,s,c=B[\'estatisticas_climatologia\'](a,\'2006-09-01\')\n        a.values[ts>pd.Timestamp(\'2006-09-01\')]=1e8\n        np.testing.assert_array_equal(cl,B[\'estatisticas_climatologia\'](a,\'2006-09-01\')[0])\n        self.assertTrue((c==30).all())\n        # A média LOO deve excluir o próprio alvo em todos os doze meses.\n        for m in range(1,13):\n            sel=a.sel(time=B[\'janela_referencia\'](\'2006-09-01\'))\n            vals=sel.values[pd.DatetimeIndex(sel.time.values).month==m]\n            np.testing.assert_allclose((s[m-1]-vals[0])/29,vals[1:].mean(0,dtype=np.float64))\n\n    def test_matriz_usa_meses_reais(self):\n        ts=pd.date_range(\'2006-01-01\',\'2007-03-01\',freq=\'MS\')\n        clima=xr.DataArray(np.ones((12,2,2),np.float32),dims=(\'month\',\'lat\',\'lon\'),\n            coords=dict(month=range(1,13),lat=[-40.,0.],lon=[-80.,-50.]))\n        at={n:xr.DataArray(np.broadcast_to((ts.month.to_numpy()+j*100)[:,None,None],(len(ts),2,2)).astype(np.float32).copy(),\n            dims=(\'time\',\'lat\',\'lon\'),coords=dict(time=ts,lat=clima.lat,lon=clima.lon))\n            for j,n in enumerate(B[\'VARIAVEIS\'])}\n        ca={n:np.zeros((12,2,2),np.float32) for n in at}\n        ca[\'_clima_indices\']=np.zeros((12,4))\n        idx=pd.DataFrame(np.repeat(ts.month.to_numpy()[:,None],4,axis=1),index=ts,columns=B[\'INDICES_NOMES\'])\n        # run_path retorna uma cópia do dict; funções conservam o namespace de definição.\n        glob=B[\'matriz_mes\'].__globals__\n        antigo=glob.get(\'INDICES_OC\')\n        glob[\'INDICES_OC\']=idx\n        try:\n            x=B[\'matriz_mes\'](at,ca,clima,\'2007-01-01\')\n            np.testing.assert_array_equal(x[0,:9],np.arange(9)*100+9)\n            np.testing.assert_array_equal(x[0,23:],np.repeat(10.,4))\n            for v in at.values():v.values[ts>pd.Timestamp(\'2006-09-01\')]=999999\n            idx.loc[idx.index>pd.Timestamp(\'2006-10-01\')]=999999\n            np.testing.assert_array_equal(x,B[\'matriz_mes\'](at,ca,clima,\'2007-01-01\'))\n        finally:\n            if antigo is None:glob.pop(\'INDICES_OC\',None)\n            else:glob[\'INDICES_OC\']=antigo\n\n    def test_pca_lag2_mascara_somente_treino(self):\n        ts=pd.date_range(\'1982-01-01\',\'2008-01-01\',freq=\'MS\')\n        rng=np.random.default_rng(13)\n        a=xr.DataArray(rng.normal(size=(len(ts),4,5)).astype(np.float32),dims=(\'time\',\'lat\',\'lon\'),\n            coords=dict(time=ts,lat=[-50.,-20.,20.,50.],lon=np.arange(5)))\n        a.values[:,:1,:1]=np.nan\n        alvos=pd.date_range(\'1982-03-01\',\'2006-09-01\',freq=\'MS\')\n        st=B[\'sst_ajustar_pca\'](a,alvos)\n        self.assertEqual(pd.Timestamp(st[\'origens_treino\'][-1]),pd.Timestamp(\'2006-07-01\'))\n        before=B[\'sst_transformar\'](a,st,[\'2007-01-01\'])\n        a.values[ts>pd.Timestamp(\'2006-11-01\')]=np.nan\n        st2=B[\'sst_ajustar_pca\'](a,alvos)\n        for k in [\'mascara\',\'componentes\',\'centro\',\'clima_mensal\']:\n            np.testing.assert_array_equal(st[k],st2[k])\n        np.testing.assert_array_equal(before,B[\'sst_transformar\'](a,st2,[\'2007-01-01\']))\n        # Validação ausente deve parar: não alargar máscara nem imputar usando futuro.\n        a.values[ts==pd.Timestamp(\'2006-11-01\'),1,1]=np.nan\n        with self.assertRaises(ValueError):B[\'sst_transformar\'](a,st,[\'2007-01-01\'])\n\n    def test_publicacao_desconhecida_bloqueia(self):\n        gate=B[\'exigir_disponibilidade_real\']\n        with self.assertRaises(ValueError):gate([dict(fonte=\'CFSv2\')],\'2026-09-30T23:59:59Z\',[\'CFSv2\'])\n        r=dict(fonte=\'SST\',disponivel_em=\'2026-09-05T00:00:00Z\',recebido_em=\'2026-09-06T00:00:00Z\',\n               versao=\'snapshot_teste\',url=\'https://example.invalid/teste\',sha256=\'0\'*64)\n        self.assertTrue(gate([r],\'2026-09-30T23:59:59Z\',[\'SST\']))\n        r[\'recebido_em\']=\'2026-10-01T00:00:00Z\'\n        with self.assertRaises(ValueError):gate([r],\'2026-09-30T23:59:59Z\',[\'SST\'])\n\n\nif __name__==\'__main__\':\n    result=unittest.TextTestRunner(verbosity=2).run(unittest.defaultTestLoader.loadTestsFromTestCase(ContratoTemporal))\n    if not result.wasSuccessful():raise RuntimeError(\'Contrato temporal falhou; não treinar.\')\n    print(\'Seis testes temporais aprovados; dados sintéticos. Sem alegação de publicação histórica.\')\n', 'protocolo.json': '{\n  "id": "nimbus_defasagens_4_3_2_v1",\n  "registrado_em_utc": "2026-09-28T19:11:53.321218+00:00",\n  "objetivo": "Contribuição pareada de oito PCs SST com entradas e chuva de treino defasadas",\n  "natureza": "retrospectivo com produtos consolidados; disponibilidade e vintages não comprovados",\n  "lags_meses": {\n    "atmosfera": 4,\n    "indices": 3,\n    "sst": 2,\n    "chuva_treino": 4,\n    "inicializacao_sazonal": 1\n  },\n  "avaliacao": [\n    "2007-01-01",\n    "2020-12-01"\n  ],\n  "holdout_independente": false,\n  "consultar_2025": false,\n  "inicio_comum": "1982-03-01",\n  "meses_maximos": 360,\n  "corte_treino": "primeiro alvo de cada bloco menos quatro meses; setembro do ano anterior",\n  "frequencia_ajuste": "uma vez a cada bloco de 24 meses; previsões mensais com entradas atualizadas",\n  "climatologias": "chuva/atmosfera/índices: 360 meses por fonte; sazonais e PCA: meses dos exemplos",\n  "pca": {\n    "componentes": 8,\n    "solver": "full",\n    "mascara_clima_centro": "somente treino",\n    "peso_area": "sqrt(cos(latitude))",\n    "padronizar_desvio": false\n  },\n  "arvores": 300,\n  "folhas": 31,\n  "taxa": 0.05,\n  "semente": 42,\n  "pontos_por_mes": 5000,\n  "ridge_alpha": 0.1,\n  "pesos": [\n    0.375,\n    0.375,\n    0.25\n  ],\n  "escalas_residuais": [\n    0.9,\n    0.875\n  ],\n  "busca_parametros": false,\n  "busca_lags": false,\n  "busca_pesos": false,\n  "metrica_primaria": "sqrt(SSE/n) sem ponderação, todos os pontos; diferença SST menos controle",\n  "secundarios": [\n    "MAE",\n    "vies",\n    "RMSE coslat",\n    "blocos",\n    "anos",\n    "meses",\n    "3 faixas de latitude"\n  ],\n  "fontes_em_producao": "data de publicação e recebimento comprovadas antes da emissão; ausência bloqueia",\n  "lags_nao_certificam_publicacao": true,\n  "cfs_publicacao_iri": "desconhecida",\n  "arquivo_sst": "ersstv5_4graus_198201_202411.nc",\n  "hash_sst": "908034f6418833302ea432f25850ad2982d1114b747b7b298e26a350eaf2891d"\n}', 'indices_noaa_197607_202009.csv': 'time_origem,nino34,nino12,tna,tsa\n1976-07-01,0.150000000,2.020000000,-0.640000000,-0.930000000\n1976-08-01,0.310000000,1.810000000,-0.270000000,-0.680000000\n1976-09-01,0.570000000,1.230000000,-0.240000000,-0.470000000\n1976-10-01,0.860000000,0.760000000,-0.250000000,-0.580000000\n1976-11-01,0.880000000,0.440000000,-0.270000000,-0.630000000\n1976-12-01,0.640000000,0.250000000,-0.600000000,-0.470000000\n1977-01-01,0.840000000,0.290000000,-0.400000000,-0.530000000\n1977-02-01,0.550000000,-0.140000000,-0.400000000,-0.270000000\n1977-03-01,0.400000000,-0.380000000,-0.270000000,-0.340000000\n1977-04-01,-0.250000000,-0.680000000,-0.250000000,-0.340000000\n1977-05-01,-0.030000000,-0.450000000,-0.280000000,-0.390000000\n1977-06-01,0.360000000,0.080000000,-0.200000000,-0.410000000\n1977-07-01,0.450000000,0.030000000,-0.420000000,-0.400000000\n1977-08-01,0.250000000,-0.520000000,-0.620000000,-0.190000000\n1977-09-01,0.440000000,-0.500000000,-0.610000000,-0.120000000\n1977-10-01,0.890000000,-0.190000000,-0.660000000,-0.310000000\n1977-11-01,1.030000000,-0.290000000,-0.570000000,-0.270000000\n1977-12-01,1.090000000,-0.560000000,-0.440000000,-0.340000000\n1978-01-01,0.720000000,-0.070000000,-0.130000000,-0.520000000\n1978-02-01,0.510000000,-0.350000000,0.020000000,-0.560000000\n1978-03-01,0.060000000,-0.770000000,0.090000000,-0.800000000\n1978-04-01,-0.370000000,-0.370000000,0.100000000,-0.960000000\n1978-05-01,-0.500000000,-0.540000000,-0.130000000,-0.990000000\n1978-06-01,-0.450000000,-0.860000000,-0.390000000,-0.780000000\n1978-07-01,-0.480000000,-0.400000000,-0.600000000,-0.330000000\n1978-08-01,-0.620000000,-0.640000000,-0.660000000,-0.400000000\n1978-09-01,-0.600000000,-0.300000000,-0.590000000,-0.500000000\n1978-10-01,-0.340000000,-0.430000000,-0.350000000,-0.610000000\n1978-11-01,-0.150000000,0.040000000,-0.300000000,-0.610000000\n1978-12-01,0.070000000,0.230000000,-0.480000000,-0.430000000\n1979-01-01,0.050000000,0.230000000,-0.080000000,-0.490000000\n1979-02-01,0.090000000,-0.580000000,0.100000000,-0.510000000\n1979-03-01,0.130000000,-0.250000000,-0.010000000,-0.650000000\n1979-04-01,0.170000000,0.510000000,0.140000000,-0.300000000\n1979-05-01,-0.090000000,0.270000000,0.150000000,-0.270000000\n1979-06-01,0.040000000,0.760000000,0.180000000,-0.210000000\n1979-07-01,-0.200000000,0.450000000,0.010000000,-0.120000000\n1979-08-01,0.210000000,0.500000000,-0.250000000,0.070000000\n1979-09-01,0.480000000,1.020000000,-0.270000000,0.140000000\n1979-10-01,0.320000000,0.670000000,-0.320000000,-0.170000000\n1979-11-01,0.350000000,0.120000000,-0.200000000,-0.120000000\n1979-12-01,0.690000000,-0.340000000,-0.030000000,-0.280000000\n1980-01-01,0.550000000,-0.100000000,0.150000000,-0.480000000\n1980-02-01,0.330000000,-0.250000000,0.230000000,-0.610000000\n1980-03-01,0.090000000,0.020000000,0.020000000,-0.610000000\n1980-04-01,0.200000000,-0.000000000,0.400000000,-0.420000000\n1980-05-01,0.130000000,0.250000000,0.500000000,-0.380000000\n1980-06-01,0.370000000,0.210000000,0.230000000,-0.390000000\n1980-07-01,0.020000000,-0.440000000,-0.080000000,-0.310000000\n1980-08-01,-0.230000000,-0.440000000,-0.040000000,-0.160000000\n1980-09-01,-0.120000000,-0.110000000,-0.170000000,-0.040000000\n1980-10-01,-0.110000000,-0.520000000,-0.380000000,-0.250000000\n1980-11-01,0.100000000,-0.680000000,-0.280000000,-0.430000000\n1980-12-01,0.360000000,-1.130000000,-0.310000000,-0.860000000\n1981-01-01,-0.460000000,-1.290000000,0.030000000,-0.610000000\n1981-02-01,-0.450000000,-0.830000000,-0.010000000,-0.730000000\n1981-03-01,-0.020000000,-0.190000000,0.450000000,-0.990000000\n1981-04-01,-0.170000000,-0.310000000,0.180000000,-0.830000000\n1981-05-01,-0.110000000,-0.220000000,0.160000000,-0.750000000\n1981-06-01,-0.150000000,-0.060000000,-0.060000000,-0.530000000\n1981-07-01,-0.430000000,-0.560000000,-0.120000000,-0.480000000\n1981-08-01,-0.180000000,-0.550000000,-0.240000000,-0.070000000\n1981-09-01,-0.070000000,-0.240000000,-0.230000000,0.050000000\n1981-10-01,-0.030000000,-0.520000000,-0.390000000,0.120000000\n1981-11-01,-0.270000000,-0.550000000,-0.180000000,0.080000000\n1981-12-01,0.070000000,-0.170000000,-0.020000000,-0.160000000\n1982-01-01,-0.040000000,-0.120000000,-0.060000000,-0.300000000\n1982-02-01,-0.130000000,-0.350000000,-0.240000000,-0.470000000\n1982-03-01,-0.020000000,-0.720000000,-0.230000000,-0.560000000\n1982-04-01,0.240000000,-0.510000000,-0.320000000,-0.660000000\n1982-05-01,0.650000000,0.080000000,-0.240000000,-0.710000000\n1982-06-01,0.920000000,0.270000000,-0.400000000,-0.760000000\n1982-07-01,0.640000000,0.600000000,-0.540000000,-0.700000000\n1982-08-01,0.930000000,1.090000000,-0.640000000,-0.770000000\n1982-09-01,1.390000000,2.010000000,-0.660000000,-0.460000000\n1982-10-01,2.000000000,2.280000000,-0.770000000,-0.330000000\n1982-11-01,2.070000000,3.080000000,-0.770000000,-0.460000000\n1982-12-01,2.330000000,3.200000000,-0.790000000,-0.430000000\n1983-01-01,2.430000000,2.730000000,-0.610000000,-0.010000000\n1983-02-01,2.220000000,2.030000000,-0.190000000,0.110000000\n1983-03-01,1.690000000,2.340000000,0.280000000,-0.480000000\n1983-04-01,1.120000000,3.360000000,0.290000000,-0.560000000\n1983-05-01,1.120000000,4.100000000,-0.030000000,-0.800000000\n1983-06-01,0.620000000,4.560000000,0.100000000,-0.760000000\n1983-07-01,-0.110000000,3.930000000,-0.050000000,-0.560000000\n1983-08-01,-0.130000000,3.110000000,-0.330000000,-0.280000000\n1983-09-01,-0.500000000,2.070000000,-0.560000000,-0.110000000\n1983-10-01,-1.030000000,1.020000000,-0.580000000,-0.310000000\n1983-11-01,-1.130000000,0.560000000,-0.260000000,-0.120000000\n1983-12-01,-0.950000000,0.220000000,-0.060000000,0.120000000\n1984-01-01,-0.800000000,-0.290000000,-0.300000000,0.350000000\n1984-02-01,-0.530000000,-0.770000000,-0.380000000,0.380000000\n1984-03-01,-0.480000000,-0.410000000,-0.250000000,0.440000000\n1984-04-01,-0.600000000,-0.250000000,-0.340000000,0.190000000\n1984-05-01,-0.640000000,-1.110000000,-0.360000000,-0.080000000\n1984-06-01,-0.900000000,-0.920000000,-0.610000000,0.140000000\n1984-07-01,-0.400000000,-0.540000000,-0.630000000,0.290000000\n1984-08-01,-0.400000000,-0.520000000,-0.620000000,0.570000000\n1984-09-01,-0.350000000,-0.360000000,-0.620000000,0.690000000\n1984-10-01,-0.830000000,-0.610000000,-0.920000000,0.360000000\n1984-11-01,-1.180000000,-0.180000000,-0.770000000,0.130000000\n1984-12-01,-1.530000000,-0.500000000,-0.790000000,-0.010000000\n1985-01-01,-0.890000000,-0.770000000,-0.680000000,-0.020000000\n1985-02-01,-0.910000000,-1.130000000,-0.550000000,0.250000000\n1985-03-01,-0.960000000,-0.590000000,-0.770000000,-0.210000000\n1985-04-01,-0.980000000,-1.380000000,-0.810000000,-0.160000000\n1985-05-01,-0.900000000,-1.410000000,-0.910000000,-0.100000000\n1985-06-01,-0.760000000,-1.250000000,-0.540000000,-0.270000000\n1985-07-01,-0.610000000,-1.040000000,-0.320000000,0.020000000\n1985-08-01,-0.490000000,-1.020000000,-0.420000000,-0.100000000\n1985-09-01,-0.570000000,-0.830000000,-0.540000000,-0.130000000\n1985-10-01,-0.640000000,-0.820000000,-0.570000000,0.050000000\n1985-11-01,-0.430000000,-0.920000000,-0.390000000,0.060000000\n1985-12-01,-0.400000000,-0.750000000,-0.540000000,0.030000000\n1986-01-01,-0.790000000,-0.590000000,-0.860000000,-0.140000000\n1986-02-01,-0.640000000,-0.350000000,-0.800000000,0.200000000\n1986-03-01,-0.510000000,-0.840000000,-0.760000000,0.040000000\n1986-04-01,-0.410000000,-0.860000000,-0.800000000,-0.030000000\n1986-05-01,-0.470000000,-0.820000000,-0.740000000,-0.080000000\n1986-06-01,-0.080000000,-0.900000000,-0.840000000,0.100000000\n1986-07-01,0.160000000,-0.140000000,-0.650000000,-0.060000000\n1986-08-01,0.240000000,0.170000000,-0.740000000,0.010000000\n1986-09-01,0.590000000,-0.130000000,-0.340000000,0.160000000\n1986-10-01,0.900000000,0.140000000,-0.630000000,-0.080000000\n1986-11-01,1.040000000,0.280000000,-0.910000000,-0.350000000\n1986-12-01,0.980000000,0.530000000,-0.980000000,-0.240000000\n1987-01-01,1.190000000,0.770000000,-0.570000000,-0.320000000\n1987-02-01,1.170000000,1.020000000,-0.350000000,-0.190000000\n1987-03-01,1.240000000,1.150000000,0.060000000,-0.180000000\n1987-04-01,0.890000000,1.510000000,0.170000000,-0.270000000\n1987-05-01,0.910000000,1.350000000,0.170000000,-0.060000000\n1987-06-01,1.240000000,1.060000000,0.370000000,0.110000000\n1987-07-01,1.340000000,1.190000000,0.210000000,0.250000000\n1987-08-01,1.480000000,0.910000000,0.220000000,0.240000000\n1987-09-01,1.530000000,1.310000000,0.020000000,0.170000000\n1987-10-01,1.360000000,1.500000000,0.020000000,0.190000000\n1987-11-01,1.290000000,0.900000000,0.050000000,0.450000000\n1987-12-01,1.050000000,0.890000000,0.220000000,0.130000000\n1988-01-01,0.690000000,0.090000000,-0.090000000,0.300000000\n1988-02-01,0.350000000,-0.270000000,-0.200000000,0.390000000\n1988-03-01,0.290000000,-0.520000000,0.120000000,0.430000000\n1988-04-01,-0.490000000,-0.900000000,0.220000000,0.280000000\n1988-05-01,-1.050000000,-1.080000000,0.080000000,0.370000000\n1988-06-01,-1.460000000,-1.960000000,0.100000000,0.420000000\n1988-07-01,-1.540000000,-1.540000000,-0.000000000,0.200000000\n1988-08-01,-1.440000000,-1.590000000,-0.310000000,0.300000000\n1988-09-01,-1.330000000,-1.430000000,-0.320000000,0.160000000\n1988-10-01,-2.090000000,-1.260000000,-0.630000000,0.150000000\n1988-11-01,-2.180000000,-0.650000000,-0.540000000,-0.050000000\n1988-12-01,-1.980000000,-0.970000000,-0.520000000,-0.070000000\n1989-01-01,-1.950000000,-0.280000000,-0.670000000,0.080000000\n1989-02-01,-1.370000000,0.050000000,-0.660000000,-0.040000000\n1989-03-01,-1.330000000,-0.450000000,-0.820000000,0.000000000\n1989-04-01,-1.110000000,-0.490000000,-1.040000000,-0.020000000\n1989-05-01,-0.800000000,-1.160000000,-0.860000000,-0.010000000\n1989-06-01,-0.640000000,-0.730000000,-0.420000000,0.000000000\n1989-07-01,-0.470000000,-0.440000000,-0.030000000,0.000000000\n1989-08-01,-0.580000000,-0.460000000,0.050000000,0.190000000\n1989-09-01,-0.380000000,-0.470000000,-0.320000000,0.060000000\n1989-10-01,-0.400000000,-0.290000000,-0.300000000,0.060000000\n1989-11-01,-0.300000000,-0.380000000,-0.350000000,-0.240000000\n1989-12-01,-0.130000000,-0.430000000,-0.070000000,-0.510000000\n1990-01-01,0.020000000,-0.330000000,-0.330000000,-0.680000000\n1990-02-01,0.360000000,-0.270000000,-0.280000000,-0.140000000\n1990-03-01,0.200000000,-0.410000000,-0.200000000,-0.030000000\n1990-04-01,0.260000000,-0.330000000,-0.100000000,0.040000000\n1990-05-01,0.320000000,-0.230000000,-0.070000000,-0.400000000\n1990-06-01,0.000000000,-0.050000000,-0.270000000,-0.230000000\n1990-07-01,0.150000000,-0.870000000,-0.220000000,-0.370000000\n1990-08-01,0.170000000,-0.610000000,-0.040000000,-0.330000000\n1990-09-01,0.120000000,-0.280000000,0.020000000,-0.220000000\n1990-10-01,0.130000000,-0.730000000,-0.010000000,-0.130000000\n1990-11-01,0.080000000,-0.550000000,0.020000000,-0.010000000\n1990-12-01,0.320000000,-0.440000000,-0.120000000,-0.040000000\n1991-01-01,0.510000000,-0.090000000,-0.390000000,-0.210000000\n1991-02-01,0.320000000,-0.100000000,-0.290000000,-0.200000000\n1991-03-01,0.100000000,0.000000000,-0.370000000,-0.210000000\n1991-04-01,0.210000000,-0.680000000,-0.540000000,-0.000000000\n1991-05-01,0.440000000,-0.010000000,-0.620000000,0.140000000\n1991-06-01,0.650000000,0.240000000,-0.670000000,0.130000000\n1991-07-01,0.620000000,0.600000000,-0.670000000,-0.200000000\n1991-08-01,0.430000000,0.000000000,-0.620000000,-0.520000000\n1991-09-01,0.220000000,0.160000000,-0.520000000,-0.510000000\n1991-10-01,0.910000000,0.420000000,-0.530000000,-0.590000000\n1991-11-01,1.140000000,0.640000000,-0.460000000,-0.520000000\n1991-12-01,1.620000000,0.450000000,-0.470000000,-0.340000000\n1992-01-01,1.610000000,0.360000000,-0.240000000,-0.610000000\n1992-02-01,1.620000000,0.440000000,-0.140000000,-0.530000000\n1992-03-01,1.500000000,0.920000000,-0.340000000,-0.740000000\n1992-04-01,1.410000000,1.720000000,-0.440000000,-0.880000000\n1992-05-01,1.270000000,1.830000000,-0.390000000,-1.010000000\n1992-06-01,0.530000000,0.710000000,-0.360000000,-1.110000000\n1992-07-01,0.270000000,-0.050000000,-0.290000000,-0.860000000\n1992-08-01,-0.150000000,-0.240000000,-0.560000000,-0.860000000\n1992-09-01,-0.140000000,-0.500000000,-0.580000000,-0.700000000\n1992-10-01,-0.350000000,-0.260000000,-0.480000000,-0.740000000\n1992-11-01,-0.140000000,-0.250000000,-0.610000000,-0.970000000\n1992-12-01,0.040000000,-0.310000000,-0.520000000,-0.630000000\n1993-01-01,0.280000000,0.030000000,-0.500000000,-0.590000000\n1993-02-01,0.420000000,0.200000000,-0.430000000,-0.390000000\n1993-03-01,0.470000000,0.530000000,-0.380000000,-0.440000000\n1993-04-01,0.920000000,0.830000000,-0.220000000,-0.110000000\n1993-05-01,0.930000000,1.130000000,-0.250000000,-0.320000000\n1993-06-01,0.640000000,0.880000000,-0.180000000,-0.340000000\n1993-07-01,0.330000000,0.320000000,-0.440000000,-0.210000000\n1993-08-01,0.160000000,0.210000000,-0.450000000,-0.160000000\n1993-09-01,0.260000000,0.120000000,-0.500000000,0.300000000\n1993-10-01,0.360000000,0.320000000,-0.560000000,0.470000000\n1993-11-01,0.280000000,-0.230000000,-0.650000000,0.570000000\n1993-12-01,0.190000000,-0.290000000,-0.700000000,0.240000000\n1994-01-01,0.040000000,-0.070000000,-0.830000000,-0.010000000\n1994-02-01,-0.160000000,-0.330000000,-0.780000000,-0.140000000\n1994-03-01,-0.060000000,-1.130000000,-0.760000000,-0.100000000\n1994-04-01,0.030000000,-1.220000000,-0.620000000,-0.130000000\n1994-05-01,0.140000000,-0.910000000,-0.700000000,-0.230000000\n1994-06-01,0.270000000,-0.590000000,-0.670000000,-0.490000000\n1994-07-01,0.160000000,-0.820000000,-0.670000000,-0.520000000\n1994-08-01,0.520000000,-0.900000000,-0.630000000,-0.370000000\n1994-09-01,0.400000000,-0.050000000,-0.620000000,-0.300000000\n1994-10-01,0.900000000,0.650000000,-0.500000000,-0.270000000\n1994-11-01,1.140000000,0.590000000,-0.340000000,-0.110000000\n1994-12-01,1.210000000,0.680000000,-0.420000000,-0.070000000\n1995-01-01,1.100000000,0.910000000,-0.300000000,0.070000000\n1995-02-01,0.870000000,0.240000000,-0.220000000,0.340000000\n1995-03-01,0.490000000,-0.570000000,-0.210000000,0.390000000\n1995-04-01,0.270000000,-0.990000000,-0.040000000,0.290000000\n1995-05-01,0.030000000,-1.010000000,0.100000000,0.240000000\n1995-06-01,0.080000000,-0.530000000,0.230000000,0.330000000\n1995-07-01,0.030000000,-0.250000000,0.310000000,0.090000000\n1995-08-01,-0.380000000,-0.520000000,0.250000000,-0.260000000\n1995-09-01,-0.570000000,-0.280000000,0.030000000,-0.190000000\n1995-10-01,-0.720000000,-0.410000000,-0.060000000,-0.270000000\n1995-11-01,-0.780000000,-0.260000000,0.200000000,-0.060000000\n1995-12-01,-0.720000000,-0.690000000,0.170000000,-0.260000000\n1996-01-01,-0.650000000,-0.610000000,0.490000000,-0.150000000\n1996-02-01,-0.660000000,-0.320000000,0.220000000,0.160000000\n1996-03-01,-0.480000000,-0.070000000,0.070000000,0.120000000\n1996-04-01,-0.340000000,-1.280000000,0.150000000,0.340000000\n1996-05-01,-0.400000000,-0.810000000,0.160000000,0.430000000\n1996-06-01,-0.120000000,-1.070000000,-0.060000000,0.560000000\n1996-07-01,-0.140000000,-0.980000000,-0.150000000,0.410000000\n1996-08-01,-0.300000000,-0.460000000,-0.360000000,0.150000000\n1996-09-01,-0.340000000,-0.530000000,-0.220000000,-0.010000000\n1996-10-01,-0.280000000,-0.800000000,-0.340000000,-0.100000000\n1996-11-01,-0.300000000,-0.790000000,-0.280000000,-0.060000000\n1996-12-01,-0.430000000,-1.030000000,-0.150000000,-0.330000000\n1997-01-01,-0.430000000,-0.560000000,0.060000000,-0.550000000\n1997-02-01,-0.240000000,-0.030000000,0.160000000,-0.600000000\n1997-03-01,-0.060000000,0.820000000,-0.000000000,-0.820000000\n1997-04-01,0.340000000,0.980000000,0.120000000,-0.810000000\n1997-05-01,0.870000000,2.160000000,0.230000000,-0.890000000\n1997-06-01,1.150000000,3.170000000,0.290000000,-0.980000000\n1997-07-01,1.600000000,3.630000000,0.060000000,-0.530000000\n1997-08-01,1.940000000,3.980000000,-0.170000000,-0.290000000\n1997-09-01,2.100000000,4.050000000,-0.120000000,0.060000000\n1997-10-01,2.290000000,4.000000000,0.160000000,0.470000000\n1997-11-01,2.420000000,4.120000000,0.270000000,0.600000000\n1997-12-01,2.300000000,4.270000000,0.150000000,0.760000000\n1998-01-01,2.420000000,3.550000000,0.190000000,0.380000000\n1998-02-01,2.080000000,2.880000000,0.590000000,0.470000000\n1998-03-01,1.490000000,2.880000000,0.590000000,0.340000000\n1998-04-01,0.900000000,3.000000000,0.430000000,0.230000000\n1998-05-01,0.680000000,3.010000000,0.390000000,0.200000000\n1998-06-01,-0.390000000,2.000000000,0.420000000,0.350000000\n1998-07-01,-0.730000000,1.310000000,0.290000000,0.590000000\n1998-08-01,-0.830000000,0.940000000,0.330000000,0.400000000\n1998-09-01,-0.820000000,0.310000000,0.130000000,0.260000000\n1998-10-01,-1.190000000,0.120000000,0.070000000,-0.010000000\n1998-11-01,-1.230000000,-0.210000000,0.120000000,0.140000000\n1998-12-01,-1.510000000,-0.140000000,0.000000000,-0.150000000\n1999-01-01,-1.530000000,-0.230000000,-0.350000000,-0.200000000\n1999-02-01,-1.410000000,0.050000000,-0.380000000,-0.220000000\n1999-03-01,-0.920000000,0.300000000,-0.380000000,-0.020000000\n1999-04-01,-0.810000000,-0.780000000,-0.280000000,0.140000000\n1999-05-01,-0.870000000,-0.430000000,-0.090000000,0.180000000\n1999-06-01,-0.950000000,-0.550000000,-0.100000000,0.180000000\n1999-07-01,-0.840000000,-0.650000000,-0.100000000,0.380000000\n1999-08-01,-0.980000000,-0.510000000,0.010000000,0.350000000\n1999-09-01,-0.840000000,-0.970000000,0.010000000,0.090000000\n1999-10-01,-1.030000000,-0.760000000,-0.180000000,0.050000000\n1999-11-01,-1.410000000,-1.110000000,-0.180000000,0.150000000\n1999-12-01,-1.540000000,-1.060000000,-0.020000000,0.100000000\n2000-01-01,-1.790000000,-0.890000000,-0.420000000,-0.120000000\n2000-02-01,-1.530000000,-0.470000000,-0.280000000,0.120000000\n2000-03-01,-1.260000000,-0.560000000,-0.090000000,0.050000000\n2000-04-01,-0.800000000,0.460000000,-0.170000000,0.060000000\n2000-05-01,-0.800000000,-0.290000000,-0.270000000,-0.050000000\n2000-06-01,-0.750000000,-0.650000000,-0.400000000,-0.110000000\n2000-07-01,-0.570000000,-0.570000000,-0.270000000,-0.100000000\n2000-08-01,-0.360000000,-0.590000000,-0.190000000,-0.000000000\n2000-09-01,-0.390000000,-0.340000000,-0.360000000,-0.130000000\n2000-10-01,-0.550000000,-0.490000000,-0.420000000,-0.120000000\n2000-11-01,-0.750000000,-0.700000000,-0.430000000,-0.110000000\n2000-12-01,-0.920000000,-0.550000000,-0.490000000,-0.270000000\n2001-01-01,-0.880000000,-0.760000000,-0.380000000,-0.170000000\n2001-02-01,-0.630000000,-0.330000000,-0.320000000,-0.100000000\n2001-03-01,-0.480000000,0.350000000,-0.010000000,-0.060000000\n2001-04-01,-0.300000000,0.420000000,-0.110000000,0.070000000\n2001-05-01,-0.300000000,-0.640000000,-0.140000000,0.120000000\n2001-06-01,-0.110000000,-0.860000000,-0.010000000,0.040000000\n2001-07-01,0.010000000,-0.700000000,0.020000000,-0.110000000\n2001-08-01,-0.070000000,-0.770000000,0.000000000,-0.060000000\n2001-09-01,-0.280000000,-1.200000000,0.130000000,-0.040000000\n2001-10-01,-0.260000000,-1.140000000,0.060000000,-0.030000000\n2001-11-01,-0.280000000,-0.940000000,0.310000000,-0.020000000\n2001-12-01,-0.460000000,-1.120000000,0.370000000,-0.250000000\n2002-01-01,-0.140000000,-0.940000000,0.530000000,-0.190000000\n2002-02-01,0.000000000,-0.300000000,0.350000000,-0.330000000\n2002-03-01,0.110000000,0.620000000,0.210000000,-0.060000000\n2002-04-01,0.140000000,0.330000000,-0.180000000,-0.000000000\n2002-05-01,0.210000000,0.180000000,-0.400000000,0.150000000\n2002-06-01,0.680000000,-0.400000000,-0.380000000,0.020000000\n2002-07-01,0.570000000,-0.440000000,-0.330000000,0.220000000\n2002-08-01,0.700000000,-0.260000000,-0.380000000,-0.190000000\n2002-09-01,0.820000000,0.040000000,-0.310000000,-0.230000000\n2002-10-01,1.160000000,0.570000000,-0.070000000,-0.090000000\n2002-11-01,1.410000000,0.780000000,-0.090000000,-0.250000000\n2002-12-01,1.410000000,0.690000000,-0.090000000,0.080000000\n2003-01-01,0.980000000,0.100000000,0.020000000,0.220000000\n2003-02-01,0.640000000,-0.210000000,-0.170000000,0.300000000\n2003-03-01,0.480000000,-0.490000000,-0.110000000,0.280000000\n2003-04-01,-0.030000000,-0.900000000,-0.150000000,0.110000000\n2003-05-01,-0.520000000,-1.350000000,-0.140000000,0.020000000\n2003-06-01,-0.190000000,-0.910000000,-0.020000000,-0.210000000\n2003-07-01,0.140000000,-0.580000000,0.090000000,0.140000000\n2003-08-01,0.050000000,0.080000000,0.260000000,0.300000000\n2003-09-01,0.150000000,-0.430000000,0.280000000,0.340000000\n2003-10-01,0.460000000,-0.070000000,0.380000000,0.310000000\n2003-11-01,0.390000000,0.290000000,0.170000000,0.360000000\n2003-12-01,0.320000000,0.310000000,0.200000000,0.300000000\n2004-01-01,0.260000000,-0.030000000,0.250000000,0.070000000\n2004-02-01,0.170000000,-0.200000000,0.340000000,-0.050000000\n2004-03-01,-0.100000000,-0.340000000,0.200000000,-0.360000000\n2004-04-01,0.060000000,-0.450000000,0.300000000,-0.570000000\n2004-05-01,0.100000000,-0.960000000,-0.030000000,-0.470000000\n2004-06-01,0.140000000,-0.690000000,0.020000000,-0.390000000\n2004-07-01,0.410000000,-0.700000000,0.180000000,-0.180000000\n2004-08-01,0.660000000,-0.440000000,0.350000000,0.020000000\n2004-09-01,0.670000000,-0.100000000,0.210000000,0.180000000\n2004-10-01,0.730000000,0.110000000,0.200000000,-0.080000000\n2004-11-01,0.620000000,0.390000000,0.320000000,0.060000000\n2004-12-01,0.710000000,0.270000000,0.240000000,0.170000000\n2005-01-01,0.560000000,0.030000000,0.220000000,0.280000000\n2005-02-01,0.260000000,-0.610000000,0.200000000,0.220000000\n2005-03-01,0.280000000,-1.150000000,0.530000000,0.260000000\n2005-04-01,0.280000000,-0.360000000,0.650000000,-0.020000000\n2005-05-01,0.300000000,0.220000000,0.810000000,-0.400000000\n2005-06-01,0.220000000,-0.150000000,0.670000000,-0.570000000\n2005-07-01,-0.010000000,-0.060000000,0.570000000,-0.570000000\n2005-08-01,-0.040000000,-0.300000000,0.410000000,-0.190000000\n2005-09-01,-0.080000000,-0.610000000,0.350000000,-0.230000000\n2005-10-01,-0.150000000,-1.050000000,0.120000000,-0.220000000\n2005-11-01,-0.440000000,-1.410000000,0.090000000,-0.260000000\n2005-12-01,-0.750000000,-1.190000000,0.260000000,-0.290000000\n2006-01-01,-0.980000000,-0.690000000,0.160000000,-0.300000000\n2006-02-01,-0.710000000,0.330000000,0.010000000,-0.040000000\n2006-03-01,-0.730000000,-0.270000000,0.020000000,0.060000000\n2006-04-01,-0.300000000,-0.890000000,0.260000000,-0.000000000\n2006-05-01,-0.110000000,-0.390000000,0.230000000,-0.110000000\n2006-06-01,0.090000000,-0.190000000,0.280000000,0.100000000\n2006-07-01,0.030000000,0.180000000,0.090000000,0.050000000\n2006-08-01,0.370000000,0.510000000,0.200000000,0.130000000\n2006-09-01,0.620000000,0.780000000,0.310000000,0.130000000\n2006-10-01,0.760000000,1.200000000,0.310000000,0.010000000\n2006-11-01,0.980000000,0.910000000,0.400000000,-0.100000000\n2006-12-01,1.100000000,0.620000000,0.180000000,0.050000000\n2007-01-01,0.590000000,0.570000000,0.210000000,-0.090000000\n2007-02-01,0.120000000,0.060000000,0.380000000,-0.060000000\n2007-03-01,-0.150000000,-0.650000000,0.170000000,-0.310000000\n2007-04-01,-0.160000000,-1.050000000,0.240000000,-0.070000000\n2007-05-01,-0.390000000,-1.470000000,0.010000000,-0.060000000\n2007-06-01,-0.160000000,-1.400000000,-0.080000000,0.120000000\n2007-07-01,-0.370000000,-0.940000000,-0.180000000,-0.020000000\n2007-08-01,-0.570000000,-1.360000000,-0.330000000,-0.040000000\n2007-09-01,-1.040000000,-1.340000000,-0.270000000,0.120000000\n2007-10-01,-1.410000000,-1.650000000,-0.190000000,-0.110000000\n2007-11-01,-1.580000000,-1.950000000,-0.070000000,-0.080000000\n2007-12-01,-1.610000000,-1.630000000,-0.210000000,-0.060000000\n2008-01-01,-1.790000000,-0.800000000,-0.100000000,-0.140000000\n2008-02-01,-1.700000000,-0.160000000,-0.000000000,0.090000000\n2008-03-01,-1.170000000,0.330000000,0.120000000,0.250000000\n2008-04-01,-0.890000000,0.500000000,-0.060000000,0.330000000\n2008-05-01,-0.640000000,-0.320000000,0.070000000,0.470000000\n2008-06-01,-0.440000000,0.260000000,0.150000000,0.390000000\n2008-07-01,-0.040000000,0.470000000,0.170000000,0.400000000\n2008-08-01,-0.040000000,0.670000000,0.120000000,0.310000000\n2008-09-01,-0.280000000,0.320000000,0.200000000,0.130000000\n2008-10-01,-0.300000000,-0.170000000,0.010000000,0.050000000\n2008-11-01,-0.370000000,-0.210000000,-0.040000000,0.130000000\n2008-12-01,-0.900000000,-0.330000000,-0.030000000,0.150000000\n2009-01-01,-1.000000000,-0.170000000,-0.170000000,0.110000000\n2009-02-01,-0.710000000,-0.310000000,-0.550000000,0.080000000\n2009-03-01,-0.720000000,-0.850000000,-0.490000000,0.200000000\n2009-04-01,-0.250000000,0.000000000,-0.470000000,0.210000000\n2009-05-01,0.170000000,0.170000000,-0.430000000,0.440000000\n2009-06-01,0.490000000,0.540000000,-0.190000000,0.280000000\n2009-07-01,0.690000000,0.700000000,0.170000000,0.140000000\n2009-08-01,0.620000000,0.660000000,0.070000000,0.040000000\n2009-09-01,0.680000000,0.220000000,-0.030000000,0.040000000\n2009-10-01,0.960000000,0.220000000,0.200000000,0.230000000\n2009-11-01,1.490000000,0.140000000,0.200000000,0.110000000\n2009-12-01,1.810000000,0.490000000,0.250000000,0.430000000\n2010-01-01,1.430000000,0.360000000,0.550000000,0.460000000\n2010-02-01,1.180000000,0.040000000,0.900000000,0.410000000\n2010-03-01,1.070000000,-0.040000000,1.120000000,0.540000000\n2010-04-01,0.560000000,0.510000000,1.250000000,0.370000000\n2010-05-01,-0.150000000,0.390000000,1.150000000,0.270000000\n2010-06-01,-0.620000000,0.140000000,0.870000000,0.540000000\n2010-07-01,-0.890000000,-1.050000000,0.740000000,0.350000000\n2010-08-01,-1.330000000,-1.230000000,0.630000000,0.290000000\n2010-09-01,-1.560000000,-1.290000000,0.370000000,0.080000000\n2010-10-01,-1.650000000,-1.530000000,0.350000000,0.210000000\n2010-11-01,-1.570000000,-1.390000000,0.390000000,0.220000000\n2010-12-01,-1.630000000,-1.030000000,0.670000000,0.150000000\n2011-01-01,-1.700000000,-0.610000000,0.730000000,0.120000000\n2011-02-01,-1.260000000,0.070000000,0.350000000,0.320000000\n2011-03-01,-0.980000000,-0.430000000,0.250000000,0.380000000\n2011-04-01,-0.740000000,-0.050000000,0.170000000,0.180000000\n2011-05-01,-0.530000000,0.250000000,0.200000000,-0.040000000\n2011-06-01,-0.250000000,0.360000000,0.350000000,-0.110000000\n2011-07-01,-0.230000000,0.270000000,0.150000000,0.020000000\n2011-08-01,-0.660000000,-0.200000000,0.150000000,0.000000000\n2011-09-01,-0.760000000,-1.010000000,-0.010000000,0.090000000\n2011-10-01,-0.930000000,-0.710000000,0.020000000,0.060000000\n2011-11-01,-1.090000000,-0.800000000,-0.100000000,-0.230000000\n2011-12-01,-1.050000000,-0.680000000,-0.060000000,-0.740000000\n2012-01-01,-0.930000000,-0.390000000,-0.090000000,-0.660000000\n2012-02-01,-0.610000000,0.400000000,-0.190000000,-0.780000000\n2012-03-01,-0.480000000,-0.020000000,-0.410000000,-0.430000000\n2012-04-01,-0.290000000,0.480000000,-0.330000000,-0.640000000\n2012-05-01,-0.180000000,0.430000000,-0.110000000,-0.380000000\n2012-06-01,0.140000000,0.990000000,0.070000000,-0.270000000\n2012-07-01,0.440000000,0.820000000,0.020000000,-0.290000000\n2012-08-01,0.660000000,0.270000000,0.190000000,-0.280000000\n2012-09-01,0.440000000,0.040000000,0.340000000,-0.130000000\n2012-10-01,0.230000000,-0.390000000,0.350000000,-0.190000000\n2012-11-01,0.330000000,-0.460000000,0.440000000,-0.160000000\n2012-12-01,-0.130000000,-0.610000000,0.260000000,-0.090000000\n2013-01-01,-0.420000000,-0.700000000,0.190000000,0.050000000\n2013-02-01,-0.400000000,-0.320000000,0.220000000,-0.020000000\n2013-03-01,-0.140000000,-0.380000000,0.550000000,-0.090000000\n2013-04-01,-0.080000000,-1.320000000,0.370000000,0.040000000\n2013-05-01,-0.280000000,-1.440000000,0.440000000,0.130000000\n2013-06-01,-0.330000000,-1.440000000,0.070000000,-0.030000000\n2013-07-01,-0.280000000,-1.360000000,0.060000000,-0.030000000\n2013-08-01,-0.290000000,-0.760000000,0.220000000,0.050000000\n2013-09-01,-0.090000000,-0.770000000,0.230000000,-0.120000000\n2013-10-01,-0.240000000,-0.160000000,0.230000000,-0.180000000\n2013-11-01,-0.020000000,-0.500000000,0.020000000,-0.260000000\n2013-12-01,-0.090000000,-0.250000000,-0.020000000,-0.160000000\n2014-01-01,-0.420000000,0.370000000,-0.120000000,-0.230000000\n2014-02-01,-0.450000000,-0.670000000,-0.250000000,-0.050000000\n2014-03-01,-0.070000000,-0.510000000,-0.500000000,-0.130000000\n2014-04-01,0.280000000,-0.250000000,-0.360000000,-0.080000000\n2014-05-01,0.450000000,0.740000000,-0.270000000,0.140000000\n2014-06-01,0.480000000,1.170000000,-0.290000000,0.150000000\n2014-07-01,0.130000000,1.040000000,-0.130000000,-0.090000000\n2014-08-01,0.140000000,0.670000000,-0.100000000,0.050000000\n2014-09-01,0.370000000,0.150000000,0.130000000,0.110000000\n2014-10-01,0.480000000,0.340000000,0.130000000,-0.060000000\n2014-11-01,0.890000000,0.570000000,0.040000000,-0.500000000\n2014-12-01,0.770000000,0.140000000,0.050000000,-0.380000000\n2015-01-01,0.590000000,-0.160000000,-0.080000000,0.100000000\n2015-02-01,0.570000000,-0.620000000,-0.060000000,0.150000000\n2015-03-01,0.480000000,-0.010000000,-0.270000000,-0.110000000\n2015-04-01,0.900000000,0.640000000,-0.400000000,0.050000000\n2015-05-01,1.040000000,1.590000000,-0.130000000,0.110000000\n2015-06-01,1.280000000,2.190000000,-0.310000000,0.070000000\n2015-07-01,1.560000000,2.320000000,-0.120000000,-0.190000000\n2015-08-01,1.870000000,2.080000000,0.090000000,-0.310000000\n2015-09-01,2.010000000,2.480000000,0.350000000,-0.190000000\n2015-10-01,2.210000000,2.380000000,0.510000000,0.060000000\n2015-11-01,2.570000000,2.210000000,0.180000000,0.190000000\n2015-12-01,2.560000000,2.090000000,0.130000000,0.440000000\n2016-01-01,2.560000000,1.820000000,0.250000000,0.470000000\n2016-02-01,2.110000000,1.090000000,0.280000000,0.320000000\n2016-03-01,1.600000000,1.100000000,0.240000000,0.250000000\n2016-04-01,1.050000000,0.290000000,0.170000000,0.080000000\n2016-05-01,0.450000000,0.450000000,0.180000000,0.010000000\n2016-06-01,0.060000000,0.520000000,0.150000000,0.370000000\n2016-07-01,-0.250000000,-0.060000000,0.100000000,0.370000000\n2016-08-01,-0.480000000,0.190000000,0.150000000,0.330000000\n2016-09-01,-0.460000000,0.250000000,0.010000000,0.110000000\n2016-10-01,-0.750000000,-0.030000000,0.120000000,0.240000000\n2016-11-01,-0.630000000,-0.080000000,0.150000000,0.100000000\n2016-12-01,-0.510000000,0.020000000,0.230000000,0.410000000\n2017-01-01,-0.340000000,0.330000000,0.240000000,0.250000000\n2017-02-01,-0.010000000,0.930000000,0.060000000,-0.180000000\n2017-03-01,-0.090000000,1.560000000,-0.030000000,-0.060000000\n2017-04-01,0.220000000,0.490000000,0.310000000,0.090000000\n2017-05-01,0.300000000,0.250000000,0.380000000,0.120000000\n2017-06-01,0.220000000,0.200000000,0.350000000,0.260000000\n2017-07-01,0.220000000,-0.460000000,0.320000000,0.100000000\n2017-08-01,-0.180000000,-0.760000000,0.330000000,0.390000000\n2017-09-01,-0.560000000,-1.280000000,0.150000000,0.120000000\n2017-10-01,-0.520000000,-1.320000000,0.030000000,0.190000000\n2017-11-01,-0.840000000,-1.440000000,0.220000000,0.020000000\n2017-12-01,-0.850000000,-1.540000000,0.250000000,-0.090000000\n2018-01-01,-0.860000000,-1.250000000,0.030000000,0.290000000\n2018-02-01,-0.730000000,-0.430000000,-0.270000000,-0.270000000\n2018-03-01,-0.730000000,-1.300000000,-0.130000000,-0.180000000\n2018-04-01,-0.360000000,-0.810000000,-0.330000000,-0.030000000\n2018-05-01,-0.120000000,-0.460000000,-0.600000000,-0.070000000\n2018-06-01,0.120000000,-0.470000000,-0.720000000,0.150000000\n2018-07-01,0.270000000,-0.080000000,-0.480000000,0.120000000\n2018-08-01,0.050000000,-0.540000000,-0.320000000,0.210000000\n2018-09-01,0.300000000,0.120000000,-0.040000000,0.210000000\n2018-10-01,0.840000000,0.300000000,-0.130000000,0.310000000\n2018-11-01,1.000000000,0.690000000,-0.320000000,0.210000000\n2018-12-01,0.970000000,0.790000000,-0.330000000,0.400000000\n2019-01-01,0.510000000,0.670000000,-0.170000000,0.410000000\n2019-02-01,0.620000000,0.660000000,-0.040000000,0.410000000\n2019-03-01,0.810000000,-0.180000000,-0.150000000,0.670000000\n2019-04-01,0.670000000,0.360000000,-0.170000000,0.340000000\n2019-05-01,0.610000000,0.240000000,-0.080000000,0.310000000\n2019-06-01,0.660000000,-0.110000000,0.060000000,0.270000000\n2019-07-01,0.410000000,-0.360000000,0.090000000,0.240000000\n2019-08-01,0.190000000,-0.290000000,0.090000000,0.220000000\n2019-09-01,0.110000000,-0.680000000,0.200000000,0.330000000\n2019-10-01,0.550000000,-0.710000000,0.010000000,0.250000000\n2019-11-01,0.740000000,0.020000000,-0.050000000,0.870000000\n2019-12-01,0.510000000,-0.190000000,-0.040000000,0.800000000\n2020-01-01,0.640000000,-0.120000000,0.010000000,0.950000000\n2020-02-01,0.480000000,-0.030000000,0.320000000,0.570000000\n2020-03-01,0.360000000,-0.440000000,0.580000000,0.310000000\n2020-04-01,0.490000000,0.030000000,0.460000000,0.410000000\n2020-05-01,-0.120000000,-0.240000000,0.320000000,0.530000000\n2020-06-01,-0.210000000,-0.430000000,0.470000000,0.290000000\n2020-07-01,-0.040000000,-1.090000000,0.380000000,0.290000000\n2020-08-01,-0.420000000,-0.430000000,0.250000000,0.300000000\n2020-09-01,-0.660000000,-0.850000000,0.150000000,0.090000000\n', 'fontes_noaa.json': '{\n  "fontes": [\n    {\n      "indice": "nino34",\n      "url": "https://psl.noaa.gov/data/timeseries/month/data/nino34.long.anom.data",\n      "sha256_bruto": "46e501b91f624ef85b525d73fbb427f9d04d30411ba6e5c7cff579b35fb4ff01"\n    },\n    {\n      "indice": "nino12",\n      "url": "https://psl.noaa.gov/data/timeseries/month/data/nino12.long.anom.data",\n      "sha256_bruto": "b13ad5ad085cd13e70ec5d82662b509d522204e38a8eefd133f14be270e51928"\n    },\n    {\n      "indice": "tna",\n      "url": "https://psl.noaa.gov/data/correlation/tna.data",\n      "sha256_bruto": "387c4125f377e57bcb683a849aefcd76da7d7f3cdcb2ac68ab086ceadf300e8b"\n    },\n    {\n      "indice": "tsa",\n      "url": "https://psl.noaa.gov/data/correlation/tsa.data",\n      "sha256_bruto": "700c0dffda0f19c1254ba482808b345eaa0538f5bf09fc3bc025f46a5fbe3a69"\n    }\n  ],\n  "captura": "2026-09-21T23:00:40.933121+00:00",\n  "extensao_197607_197611": true,\n  "sobreposicao_exata": true,\n  "vintages": false\n}', 'hashes_oficiais.json': '[\n  {\n    "nome": "treino_tp.nc",\n    "sha256": "012bebbe0e38aefdf4f9e2432c3ca22f71d94ea1a1b5421da9d9e5048632cab0",\n    "bytes": 186685299\n  },\n  {\n    "nome": "treino_tp_alvo.nc",\n    "sha256": "915f67f2627e37399830d1dd92b209ce1160082b0645662675eab17df957abe4",\n    "bytes": 186509303\n  },\n  {\n    "nome": "teste_features.nc",\n    "sha256": "70bcb5c9bb9c011c8901660834b42dc57ff7e54a4f34161d489ec5e9f05c1f23",\n    "bytes": 39057097\n  },\n  {\n    "nome": "sample_submission.csv",\n    "sha256": "34fe449ddb309b0579a87786f76049863833d82c26f183d18e943dc76f1e4c8c",\n    "bytes": 48145117\n  },\n  {\n    "nome": "treino_cloud_cover.nc",\n    "sha256": "607ee1210615a0ca8a5c954737617e668a5f016eb881554cb1468fda74b4fde1",\n    "bytes": 190610590\n  },\n  {\n    "nome": "treino_geopotential_850.nc",\n    "sha256": "ca3baca0177e2e90b42901803f348a4b4affe88bee202b76af3f293fa7035c17",\n    "bytes": 159707312\n  },\n  {\n    "nome": "treino_rel_hum_850.nc",\n    "sha256": "90f62b44b1b461259e7539249914e26cf09993f3bcd572ba4871931e45264b19",\n    "bytes": 180398147\n  },\n  {\n    "nome": "treino_shum_850.nc",\n    "sha256": "35cf8071b68d1e924ad87e0f8bd801d0a66d1d78ae75fc3cb623bde85c3f47bb",\n    "bytes": 175651693\n  },\n  {\n    "nome": "treino_surface_pressure.nc",\n    "sha256": "f00a5e44a03ea923ac0f693b975ef71cd347f8ad3d07692c602e3ac9a25c10fb",\n    "bytes": 160228025\n  },\n  {\n    "nome": "treino_t2.nc",\n    "sha256": "7d8c53b438289cddf87a11414ad10c8167f52ff33c6eff459a3288975079ef23",\n    "bytes": 175952066\n  },\n  {\n    "nome": "treino_temperature_850.nc",\n    "sha256": "1db7b4fdeba87b6a25f7fb6f1e40abec50964234dd9cc918cd34f902b8e41bac",\n    "bytes": 172474226\n  },\n  {\n    "nome": "treino_u_850.nc",\n    "sha256": "73c898f27bea25cf2ab0914bbe5c29acc9794a93f8b6b3affa39854bebccf761",\n    "bytes": 188451602\n  },\n  {\n    "nome": "treino_v_850.nc",\n    "sha256": "7bab583972e5718bb0e0fe0fe6a056bd56c968af1f1497205182f7621e26c241",\n    "bytes": 197354831\n  }\n]', 'proveniencia.json': '[\n  {\n    "arquivo": "entregas\\\\banca_8G\\\\WorCAP_modelo_hibrido_reproducao.py",\n    "sha256": "547bd7ac54c8873b5d6e237627f4b6dba94851259548a455f8574e2ecfff8616"\n  },\n  {\n    "arquivo": "pesquisa\\\\sst_pca\\\\extensao_sst.py",\n    "sha256": "789a8a2d38cca97ae32d04db7aa11aecc0c175dacd31bd84ab465ff8eff5d8d8"\n  },\n  {\n    "arquivo": "pesquisa\\\\disponibilidade_operacional\\\\defasagens.py",\n    "sha256": "89631394f19f08b6a41cfce48154663cc0d31dc7d61e3276095be23c116d931f"\n  },\n  {\n    "arquivo": "pesquisa\\\\disponibilidade_operacional\\\\executar_comparacao.py",\n    "sha256": "70c2cd10f7b7dbe9f3a1bde8bc83baa9c9ff215949f8e9a212c5212c9948da4a"\n  },\n  {\n    "arquivo": "pesquisa\\\\disponibilidade_operacional\\\\testar_contrato.py",\n    "sha256": "c50496095b0300c616e78f645a204bcadcc817281ee091074baf240639e03e18"\n  }\n]', 'LEIA_ME.md': '# Nimbus PA — fontes defasadas e SST global\n\nEste notebook compara dois híbridos de precipitação mensal com o mesmo calendário de informações.\nUm utiliza atmosfera, índices oceânicos, SEAS5 e CFSv2; o outro acrescenta **oito componentes principais\nda temperatura da superfície do mar (SST)** aos dois componentes LightGBM.\n\nO objetivo é verificar se a SST ainda contribui quando os produtos mensais não são tratados como\ndisponíveis imediatamente ao fechar o mês. É uma etapa de pesquisa, com arquivos consolidados.\n\n## Calendário fixado antes do treinamento\n\n| Informação para prever o mês T | Mês usado | Exemplo: janeiro/2007 |\n|---|---|---|\n| Atmosfera ERA5 consolidada | T−4 | Setembro/2006 |\n| Índices NOAA derivados de HadISST | T−3 | Outubro/2006 |\n| SST ERSSTv5 | T−2 | Novembro/2006 |\n| Inicialização SEAS5/CFSv2 | T−1 | Dezembro/2006 |\n| Chuva mais recente permitida no treinamento | T−4 do primeiro alvo do bloco | Setembro/2006 |\n\nA avaliação cobre **sete blocos de 24 meses entre 2007 e 2020**. Em cada bloco, o modelo fica fixo;\nas entradas são atualizadas mês a mês, usando o calendário acima. Isso não significa emitir todas\nas 24 previsões em uma única data. O treinamento termina em setembro anterior ao início do bloco,\ncom no máximo 360 meses e início comum em março/1982. A primeira SST é janeiro/1982.\n\nAs climatologias de chuva, atmosfera e índices usam janelas móveis de 360 meses, cada qual nas\ndatas reais de sua fonte. SEAS5, CFSv2 e PCA usam os meses associados aos exemplos disponíveis.\nA feature climatológica da chuva e o resíduo de treino excluem a própria observação (leave-one-out).\n\n## Modelo e comparação\n\n- Dois LightGBMs: 300 árvores, 31 folhas, taxa 0,05, semente 42 e 5.000 pontos por mês.\n- Regressão ridge local entre as duas previsões sazonais, penalização 0,1.\n- Mistura fixa: 37,5% de cada árvore e 25% da regressão. Escalas dos resíduos: 0,9 e 0,875.\n- A regressão e as colunas originais de treinamento são idênticas entre os dois tratamentos.\n- PCA: oito componentes, SVD completa, anomalias mensais, peso sqrt(cos(latitude)), sem padronizar\n  por desvio de cada pixel. Máscara oceânica, climatologias, centro e componentes são ajustados só no treino.\n- Não há busca de hiperparâmetros, pesos, quantidade de PCs ou defasagens nesta execução.\n\nCada versão é treinada novamente. Avaliamos todos os 78.561 pontos em todos os 168 meses:\n13.198.248 erros por modelo. A métrica principal é sqrt(SSE/n), em mm/dia. MAE, viés, RMSE com\npeso de área, meses, anos, blocos e faixas de latitude ajudam a localizar ganhos e perdas.\nOs modelos ajustam e avaliam precipitação ERA5, uma reanálise, não pluviômetros independentes.\n\n## O que a política resolve e o que ainda falta\n\nAs defasagens são **hipóteses conservadoras, não comprovantes de publicação**. ERA5T mensal chega\ncerca de cinco dias após o mês; ERA5 final substitui o preliminar aproximadamente dois meses depois.\nUsar T−4 deixa margem para a consolidação. HadISST adiciona o mês anterior ao último no início\nde cada novo mês; T−3 deixa margem adicional. ERSST T−2 é uma hipótese de disponibilidade a\nconferir em cada emissão, sem garantia universal de data neste notebook.\n\nSEAS5 via C3S tem agenda nominal de publicação no dia 6, 12 UTC. **A agenda C3S não comprova\na publicação do agregado CFSv2 no endpoint IRI usado aqui.** Seus membros têm inicializações\nanteriores a T, porém falta um arquivo histórico das datas de chegada desse agregado.\n\nOs dados finais podem ter revisões posteriores e os hindcasts foram produzidos retrospectivamente.\nPortanto, a comparação mede sensibilidade às defasagens, e não uma operação histórica certificada.\nUma futura emissão deverá comprovar a chegada e a versão de cada arquivo antes do instante\nde previsão; a função de verificação bloqueia datas ausentes ou posteriores. Este notebook\n**não emite previsão de produção**. A autenticação dos registros exige um coletor auditável.\n\n2007–2020 já foi usado para desenvolver modelos; não é reserva inédita. O teste anterior de 2025\npermanece arquivado e **não é carregado nesta etapa**. Não há decisão baseada no placar Kaggle.\n\n## Execução no Kaggle\n\nAnexe os mesmos quatro Inputs: dados oficiais WorCAP, `worcap-seas5-dados`, `worcap-cfsv2-dados`\ne o snapshot que contém `ersstv5_4graus_198201_202411.nc`. Não precisa de token nem novos downloads.\nUse CPU e o ambiente da referência: NumPy 2.0.2, pandas 2.3.3, xarray 2025.12.0, LightGBM 4.6.0.\n\nExecute todas as células em ordem. A primeira materializa código e protocolo, a segunda executa\ntestes temporais sintéticos, e a seguinte treina e avalia. Cada bloco grava modelos, PCA,\nclimatologias, coeficientes ridge, previsões e auditorias. Retomar a mesma pasta exige hashes\nidênticos de código, dados, ambiente e arquivos de cada bloco já concluído.\n\nOs resultados ficam em `/kaggle/working/nimbus_defasagens_4_3_2`. Salvar uma versão concluída\ncom outputs preserva modelos e previsões; o ZIP de relatórios contém tabelas, documentação e auditorias.\nOs artefatos da competição e o notebook anterior não são alterados.\n\n## Fontes e rastreabilidade\n\n- [ERA5 — documentação ECMWF](https://confluence.ecmwf.int/spaces/CKB/pages/76414402/ERA5%2Bdata%2Bdocumentation).\n- [HadISST — Met Office](https://www.metoffice.gov.uk/hadobs/hadisst/).\n- [ERSST — NOAA NCEI](https://www.ncei.noaa.gov/products/extended-reconstructed-sst).\n  Mantemos ERSSTv5 por comparabilidade, embora a página atual já apresente v6.\n- [C3S — dados e calendário](https://confluence.ecmwf.int/spaces/CKB/pages/104239050/Summary%2Bof%2Bavailable%2Bdata).\n- [CFSv2 — IRI Data Library](https://iridl.ldeo.columbia.edu/SOURCES/.Models/.NMME/.NCEP-CFSv2/.HINDCAST/.PENTAD_SAMPLES_FULL/.prec/).\n\nFontes documentais consultadas em 28/09/2026. Os manifests existentes conferem produto, unidade,\ngrade e membros sazonais. Os cinco meses NOAA julho–novembro/1976 são recuperados dos arquivos\nbrutos preservados; o período sobreposto é conferido exatamente contra o snapshot original.\nNão se atualiza silenciosamente nenhuma série ou versão de dados.\n'}
HASHES_CODIGO = {'biblioteca.py': 'df68d363b952ecb885cec67111fb795857f4c1d419a3940ad0f101b23015706c', 'executar_comparacao.py': '70c2cd10f7b7dbe9f3a1bde8bc83baa9c9ff215949f8e9a212c5212c9948da4a', 'testar_contrato.py': 'c50496095b0300c616e78f645a204bcadcc817281ee091074baf240639e03e18', 'protocolo.json': '5a531778786b49ca9cd8ac337cdb3297ac580d67f78d5f278492e41ff6aa243a', 'indices_noaa_197607_202009.csv': '69226b2e65f783b4d4971a254d1e325d0908447c76ec1df3bf532ecef1f0cf5b', 'fontes_noaa.json': '91f412b2e5ccd5c273e2543158fc2cdf72013828907edd03469f13c704fd8240', 'hashes_oficiais.json': 'bd5648197c4cf25f85d89a32c701b422f1e0b788421766cc7b8aca99ea42223d', 'proveniencia.json': '226ee5aba1021624d97170862b88dbb3feafab10ee1f2a12a2338108a83fc20b', 'LEIA_ME.md': 'b46957f9c5d512dc94f70745a439d18b918ac149becd6b36f7eef799402e2851'}
for nome, conteudo in ARQUIVOS.items():
    assert hashlib.sha256(conteudo.encode()).hexdigest() == HASHES_CODIGO[nome]
    caminho = CODIGO/nome
    if caminho.exists():
        assert caminho.read_text(encoding='utf-8') == conteudo, 'Código diferente; preservar execução: ' + nome
    else:
        caminho.write_text(conteudo, encoding='utf-8')
print('Código e protocolo registrados. Nenhum modelo ajustado nesta célula.')
print('Protocolo SHA-256:', HASHES_CODIGO['protocolo.json'])


## 1. Testes temporais antes dos dados reais
Testa virada do ano, corte da chuva, meses das fontes, PCA e bloqueio quando a chegada é desconhecida.

In [ ]:
_TESTES = runpy.run_path(str(CODIGO/'testar_contrato.py'), run_name='__main__')

## 2. Comparação pareada
Quatro LightGBMs e uma regressão local por bloco. Somente arquivos de desenvolvimento são carregados. Pesos, escalas e defasagens ficam fixos. Relatórios parciais são preservados por bloco.

In [ ]:
RESULTADO = runpy.run_path(str(CODIGO/'executar_comparacao.py'), run_name='__main__')

## 3. Resultado e arquivos
As tabelas discriminam o efeito de SST sob a mesma política temporal. Não comparam isoladamente o custo de cada defasagem nem certificam operação real.

In [ ]:
from IPython.display import display, Markdown, Image, FileLink
assert (RAIZ/'resultado.md').is_file(), 'Comparação ainda não concluída.'
display(Markdown((RAIZ/'resultado.md').read_text(encoding='utf-8')))
display(Image(filename=str(RAIZ/'comparacao.png')))
display(FileLink('nimbus_defasagens_4_3_2/relatorios.zip'))

In [ ]:
"""Conferência independente dos relatórios exportados, sem ajustar modelos."""
from pathlib import Path
import hashlib
import json
import sys
import numpy as np
import pandas as pd


def conferir(raiz):
    raiz = Path(raiz)
    inventario = json.loads((raiz/'inventario.json').read_text(encoding='utf-8'))
    verificados = 0
    for item in inventario:
        p = raiz/item['arquivo']
        if not p.is_file():
            # O ZIP de relatórios omite modelos e grandes arrays, conforme documentado.
            assert p.suffix in {'.txt', '.nc', '.npz'}, item['arquivo']
            continue
        assert p.stat().st_size == item['bytes'], p
        assert hashlib.sha256(p.read_bytes()).hexdigest() == item['sha256'], p
        verificados += 1
    d = pd.read_csv(raiz/'metricas_mensais_regioes.csv')
    assert not d.duplicated(['modelo','mes_alvo','regiao']).any()
    assert set(d.modelo) == {'controle_defasado','sst_defasada','climatologia'}
    assert len(d) == 3*168*4
    assert set(pd.to_datetime(d.mes_alvo)) == set(pd.date_range('2007-01-01','2020-12-01',freq='MS'))
    cols = ['n','sse','soma_erro','soma_erro_absoluto','sse_area','peso_area']
    reg = d[d.regiao != 'dominio_inteiro'].groupby(['modelo','mes_alvo'])[cols].sum().sort_index()
    dom = d[d.regiao == 'dominio_inteiro']
    assert np.allclose(reg, dom.set_index(['modelo','mes_alvo'])[cols].sort_index(), rtol=1e-12, atol=1e-7)
    for nome, eixo, dados in [('globais',['modelo'],dom),('blocos',['modelo','bloco'],dom),
                              ('anos',['modelo','ano'],dom),('regioes',['modelo','regiao'],d)]:
        soma = dados.groupby(eixo)[cols].sum().sort_index()
        salvo = pd.read_csv(raiz/f'metricas_{nome}.csv').set_index(eixo).sort_index()
        assert soma.index.equals(salvo.index)
        assert np.allclose(soma, salvo[cols], rtol=1e-12, atol=1e-7), nome
        for coluna, esperado in [('rmse',np.sqrt(soma.sse/soma.n)),
            ('mae',soma.soma_erro_absoluto/soma.n),('vies',soma.soma_erro/soma.n),
            ('rmse_area',np.sqrt(soma.sse_area/soma.peso_area))]:
            assert np.allclose(esperado,salvo[coluna],rtol=1e-12,atol=1e-12), (nome,coluna)
    globais = pd.read_csv(raiz/'metricas_globais.csv').set_index('modelo')
    assert (globais.n == 13198248).all()
    for bloco in ['H1','H2','H3','H4','A','B','C']:
        pasta = raiz/bloco
        cal = pd.read_csv(pasta/'calendario_emissoes.csv')
        t = pd.to_datetime(cal.time_alvo).dt.to_period('M')
        for coluna,lag in [('atmosfera',4),('indices',3),('sst',2),('inicializacao_sazonal',1)]:
            assert (pd.to_datetime(cal[coluna]).dt.to_period('M') == t-lag).all()
        corte = pd.to_datetime(cal.ultimo_alvo_treino).dt.to_period('M')
        assert (corte <= t.iloc[0]-4).all()
        aud = json.loads((pasta/'auditoria.json').read_text())
        assert aud['meses'] <= 360
        for chave in ['chuva_pos_corte_invariante','atmosfera_apos_Tmenos4_invariante',
                      'indices_apos_Tmenos3_invariantes','sst_apos_Tmenos2_invariante',
                      'climas_sazonais_somente_treino']:
            assert aud[chave] is True, (bloco,chave)
    resumo = json.loads((raiz/'resumo.json').read_text())
    assert resumo['alvos_2025_lidos'] is False and resumo['producao_liberada'] is False
    delta = globais.loc['sst_defasada','rmse']-globais.loc['controle_defasado','rmse']
    assert np.isclose(delta,resumo['delta_rmse'],rtol=1e-12,atol=1e-12)
    resultado = dict(arquivos_sha256_conferidos=verificados,linhas_mensais=len(d),
        agregacoes_conferidas=True,calendarios_conferidos=True,auditorias_conferidas=True,
        delta_rmse=float(delta),sem_novo_ajuste=True)
    (raiz/'conferencia_independente.json').write_text(json.dumps(resultado,indent=2),encoding='utf-8')
    print(json.dumps(resultado,indent=2))
    print(globais[['rmse','mae','vies','rmse_area']].to_string())



conferir(RAIZ)
